# Single-Task Activity Classification Pipeline for PSMA Inhibitors (F6)

Predicts whether a compound is an **active** PSMA (glutamate carboxypeptidase II / FOLH1)
inhibitor, and ranks a screening library by P(active). It keeps the transfer-learning
trunk, the three-model committee and the applicability-domain machinery of F3–F5, but
changes the prediction target and the evaluation protocol in response to four findings
from the F4/F5 runs.

## Why this is not the F5 pipeline

**1. Random splits were measuring the wrong thing.** The 681-compound BindingDB PSMA set
is a handful of congeneric families, so random 80/20 partitions put close analogues on
both sides. Held-out R² for pIC50 fell from 0.57 (random) to 0.42 (scaffold-grouped) to
**0.17 ± 0.27** (Butina-cluster-grouped). Roughly two thirds of the original headline
number was analogue leakage.

**2. That ±0.27 was bimodal, not noisy.** Butina cluster 0 holds 94 compounds — the
single largest family in the set — and **89 of them come from one patent,
US20250213705**, which supplies 119 of the 681 compounds overall. When
`GroupShuffleSplit` happened to place that cluster in the test half, the model lost a
quarter of its labels *and* faced a family it had no analogue for; those repeats
collapsed while the rest were unremarkable. Averaging the two regimes produced a mean
that described neither. **Part E.2 therefore replaces repeated grouped draws with
leave-one-cluster-out**, reporting each family separately so "generalises to A, B, C and
fails on D" can be stated instead of averaged away.

**3. pKi was largely not an independent measurement.** Of the 169 compounds measured both
ways, 111 had pKi − pIC50 fixed at 2.145 log units (IC50/Ki = 140.0 exactly, to within
reporting precision) and all came from that same patent; a further 28 shared a fixed
ratio of 5.0 from a second source. Those Ki values were derived from the IC50 values by
Cheng–Prusoff, not measured. Three consequences: the 1.4-unit offset between the two
endpoint distributions is an artifact rather than pharmacology; the masked multi-task
loss was giving dual-labelled compounds twice the weight of single-labelled ones, so one
patent accounted for ~27% of the gradient signal from 17.5% of the molecules; and pKi's
consistently *better* scores across every split scheme (0.583 vs 0.566 random, 0.484 vs
0.420 scaffold) are explained by derived labels carrying no independent measurement
noise. **The pipeline is therefore single-task on pIC50.** Part A.2b retains the
diagnostic that establishes this, and Part A.2c re-derives the dataset accordingly.

**4. The reported uncertainty was ~4× too tight.** Every compound in F5's top-15 sat at
Tanimoto dissimilarity 0.73–0.83 — unfamiliar chemistry, the regime where measured error
was ~1.2 log units — yet shipped with ±0.3 error bars, because Optuna had tuned dropout
to 0.047 and left MC-dropout with almost no variance to report. A predicted *probability*
is checkable against observed frequency in a way an error bar is not, which is the second
reason for the switch to classification.

## What this pipeline does

1. **Pretrain** a shared trunk on 6,547 ChEMBL bioactivities from related zinc
   metallopeptidases (regression on a generic potency label), exactly as before.
2. **Fine-tune** a binary classification head on the PSMA set, with class-weighted
   cross-entropy.
3. **Recover the censored data.** F5's cleaning discarded every `>`/`<` measurement
   because regression cannot fit a bound. Classification can: `IC50 > 100 µM` is a
   confident *inactive* label. This adds ~32 compounds to the class the model most
   lacked.
4. **Three model families** — transfer NN, LightGBM, and PLS-DA on a Nystroem kernel map
   — combined by **equal-weight probability averaging**. F5's fitted blend weights lost
   to a plain average in all six scheme × endpoint comparisons, so the weight optimizer
   is gone.
5. **Leave-one-cluster-out evaluation** with per-family results, plus a random-split
   reference for comparison with F5, plus the paired pretraining ablation.
6. **Screening metrics that match the use case**: ROC-AUC and PR-AUC, and enrichment
   factor plus precision@N — "of my top 200 picks, how many are really active" is the
   number that decides whether a shortlist earns lab time, and no earlier version
   computed anything like it.
7. **Applicability domain with Tanimoto as a veto.** F5's consensus rule
   (`n_flags_OOD >= 3`) let a compound pass while failing the structural-similarity test
   — the one that matters for a congeneric target. Every F5 top hit was flagged
   `tanimoto_OOD` and passed anyway. `CONFIG.ad_tanimoto_veto` (default on) makes that
   flag decisive.

## Pre-registered thresholds

Set before any model was fitted, anchored on pharmacology rather than on quantiles, and
checked against the distribution in Part A.2b:

- **active: pIC50 ≥ 8.0** (IC50 ≤ 10 nM)
- inactive: pIC50 < 8.0, or a censored bound proving pIC50 < 8.0

The observed pIC50 distribution is a single broad left-skewed hump (mixture BIC prefers
one component; the density scan found no valleys), mean 7.086, s.d. 1.459, with p25 =
6.085 and p75 = 8.194. The 10 nM cut therefore sits close to the upper quartile — both
pharmacologically meaningful and reasonably balanced, giving 128 actives against 356
non-actives once the censored rows are folded in. A three-class
(active / moderate / inactive) variant at 8.0 / 6.0 is available via
`CONFIG.n_activity_classes = 3`, which is viable for pIC50 (128 / 215 / 141) though not
for pKi (30 inactives), and is not the default.

## What this does not fix

The screening library is still the fragment-contaminated file — its first entries are
butyl chloride, β-propiolactone and diethylene glycol. LigPrep variants of the same
compound are still not merged. Compound deduplication still happens on raw SMILES before
RDKit standardisation. And nothing here has been tested in a lab. Classification makes
the output honest about the model's resolution; it does not make the model better, and
the leave-one-cluster-out numbers should be expected to show that plainly.

## 0. Setup

Install dependencies. Safe to re-run — pip skips anything already satisfied.


In [ ]:
%pip install -q duckdb rdkit pandas numpy scikit-learn matplotlib seaborn requests tqdm scipy joblib \
    lightgbm optuna scikit-optimize torch chembl_webresource_client


In [ ]:
import os
import io
import re
import json
import zipfile
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Sequence, List, Dict, Tuple

import duckdb
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs

from sklearn.model_selection import train_test_split, KFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import make_scorer
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

import torch
import torch.nn as nn

import optuna
from skopt import BayesSearchCV
from skopt.space import Integer, Real

from lightgbm import LGBMRegressor

import matplotlib.pyplot as plt
import seaborn as sns

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="notebook")

np.random.seed(0)
torch.manual_seed(0)

# ---- GPU support -----------------------------------------------------------
# Every NN training/inference function below (`pretrain_trunk`, `finetune_multitask`,
# `predict_with_mc_dropout`) accepts this `DEVICE` and moves its model/tensors onto it,
# so the entire pretrain -> fine-tune -> MC-dropout pipeline runs on GPU automatically
# when one is available (CUDA, or Apple Silicon MPS), and falls back to CPU otherwise.
# On Colab: Runtime -> Change runtime type -> select a GPU, then re-run from the top.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU detected: {torch.cuda.get_device_name(0)} -> using CUDA.")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Apple Silicon GPU detected -> using MPS.")
else:
    DEVICE = torch.device("cpu")
    print("No GPU detected -> using CPU. (Colab: Runtime > Change runtime type > GPU.)")


def rmse(y_true, y_pred) -> float:
    '''Root-mean-squared-error, computed manually so this works across sklearn versions
    (newer sklearn removed mean_squared_error's `squared=False` argument).'''
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


print("Environment ready.")


## 1. Configuration

Everything that changes between projects lives here: ChEMBL pretraining scope, the
BindingDB target/endpoints, screening library path, AD thresholds, model
architectures, and every hyperparameter-search setting. Nothing below this cell
needs to change for a routine re-run.


In [ ]:
@dataclass
class Config:
    # ---- ChEMBL pretraining set (domain-targeted by default) -------------------
    # A bioactivity set used only to *pretrain* the shared NN trunk (transfer
    # learning) before it ever sees PSMA data. Pulled live via
    # chembl_webresource_client; cached to `chembl_pretrain_cache` so re-runs are free.
    #
    # "domain_targeted" (default): restrict pretraining to targets related to PSMA —
    # PSMA (GCPII/FOLH1) is a zinc-dependent, co-catalytic-dinuclear-zinc metallopeptidase
    # (MEROPS clan MH, family M28). Other zinc metallopeptidases/metalloproteases are
    # the natural "related" pool: their inhibitors are built around the same handful of
    # zinc-binding chemotypes PSMA inhibitors use (phosphonates, thiols, hydroxamates,
    # ureas), so pretraining here teaches the trunk that specific structure-activity
    # vocabulary rather than generic, unrelated bioactivity patterns.
    # "broad": the original target-agnostic sample across arbitrary human targets.
    chembl_pretrain_mode: str = "domain_targeted"     # "domain_targeted" | "broad"
    chembl_pretrain_target_keywords: Sequence[str] = field(default_factory=lambda: [
        "glutamate carboxypeptidase",       # PSMA's own family (paralogs, e.g. NAALADase L)
        "aminopeptidase",                   # other M-clan zinc peptidases, same catalytic chemistry
        "leucine aminopeptidase",
        "carboxypeptidase",
        "angiotensin-converting enzyme",    # zinc metallopeptidase; classic phosphonate/thiol ZBGs
        "neprilysin",                       # zinc metallopeptidase (NEP), thiol/phosphonate inhibitors
        "endothelin-converting enzyme",     # related M13 zinc metallopeptidase
        "thermolysin",                      # archetypal zinc metallopeptidase used to design ZBGs
        "matrix metalloproteinase",         # zinc metalloproteinases, hydroxamate ZBGs
        "insulin-degrading enzyme",         # zinc metallopeptidase
        "meprin",                           # zinc metalloproteinase
        "carbonic anhydrase",               # different fold, but a classic zinc metalloenzyme drug
                                             # target with a rich, structurally-diverse zinc-binding
                                             # inhibitor literature (sulfonamides, etc.)
    ])
    chembl_pretrain_target_type: str = "SINGLE PROTEIN"
    chembl_pretrain_min_targets_or_fallback_to_broad: int = 5   # if fewer targets match, fall back to "broad"
    chembl_pretrain_cache: str = "./ad_pipeline_data/chembl_pretrain_domain.csv"
    chembl_pretrain_standard_types: Sequence[str] = field(
        default_factory=lambda: ["IC50", "Ki", "Kd", "EC50"]
    )
    chembl_pretrain_organism: str = "Homo sapiens"
    chembl_pretrain_max_targets: int = 150      # cap on distinct targets sampled ("broad" mode only)
    chembl_pretrain_per_target_cap: int = 300   # max activities pulled per target
    chembl_pretrain_max_compounds: int = 10000   # hard cap on the final pretraining set
    chembl_pretrain_endpoint_name: str = "pX_generic"   # generic potency label used only for pretraining
    force_rebuild_chembl_cache: bool = False

    # ---- BindingDB acquisition (PSMA fine-tuning set) --------------------------
    bindingdb_tsv_zip_name: str = "BindingDB_All_202609_tsv.zip"
    bindingdb_download_url: str = (
        "https://www.bindingdb.org/rwd/bind/downloads/BindingDB_All_202609_tsv.zip"
    )
    auto_resolve_bindingdb_url: bool = True
    data_dir: str = "./ad_pipeline_data"
    force_redownload: bool = False

    # ---- Target / multi-task endpoint selection (BindingDB SQL filter) --------
    target_uniprot_id: Optional[str] = "Q04609"          # PSMA / FOLH1 / GCPII
    target_name_like: Optional[str] = "Glutamate carboxypeptidase 2"
    organism_like: Optional[str] = "Homo sapiens"

    # Multi-task: model BOTH endpoints jointly instead of picking one.
    affinity_columns: Sequence[str] = field(default_factory=lambda: ["IC50 (nM)", "Ki (nM)"])
    endpoint_names: Sequence[str] = field(default_factory=lambda: ["pIC50", "pKi"])

    max_affinity_nm: float = 1e6
    min_affinity_nm: float = 1e-3

    # ---- Curated screening library ----------------------------------------------
    screening_library_path: str = "/Enamine_screen_library.sdf"
    screening_smiles_col: str = "SMILES"
    screening_id_col: Optional[str] = "compound_id"

    # ---- Featurization ------------------------------------------------------------
    morgan_radius: int = 2
    morgan_n_bits: int = 2048
    feature_set: str = "fp_plus_descriptors"   # "fp_plus_descriptors" | "descriptors_only" | "fp_only"

    # ---- Applicability domain thresholds (calibrated on the PSMA fine-tuning set) --
    ad_percentile_threshold: float = 95.0
    unfamiliarity_k: int = 5
    latent_pca_components: int = 20

    # ---- Neural-network transfer-learning model ------------------------------------
    nn_hidden_dims: Sequence[int] = field(default_factory=lambda: (256, 128))
    nn_dropout: float = 0.2
    nn_lr: float = 1e-3
    nn_finetune_lr: float = 5e-4
    nn_weight_decay: float = 1e-5
    nn_pretrain_epochs: int = 40
    nn_finetune_total_epochs: int = 80
    nn_freeze_epochs: int = 15          # epochs of head-only training before unfreezing the trunk
    nn_batch_size: int = 32
    nn_pretrain_batch_size: int = 128
    nn_mc_dropout_passes: int = 30      # forward passes for MC-dropout uncertainty

    # ---- LightGBM baseline (trained directly on BindingDB, one model per task) -----
    n_bootstrap: int = 30               # bootstrap ensemble size for GBM uncertainty

    # ---- Hyperparameter search ------------------------------------------------------
    run_gridsearch: bool = True
    run_bayes_search: bool = True       # scikit-optimize BayesSearchCV
    run_optuna: bool = True             # Optuna TPE search for the NN
    bayes_search_n_iter: int = 20
    optuna_n_trials: int = 25
    optuna_cv_folds: int = 3            # cheap inner CV used inside the Optuna objective

    test_size: float = 0.2
    cv_folds: int = 10
    random_state: int = 42
    n_eval_repeats: int = 10          # repeated random splits for robust held-out statistics
    n_calibration_bins: int = 5       # uncertainty-calibration reliability diagram bins

    # ---- Output ----------------------------------------------------------------------
    output_dir: str = "./ad_pipeline_outputs"


    # ================================================================================
    # F6: single-task activity classification
    # ================================================================================
    # The data pull below still fetches BOTH endpoints, because Part A.2b's diagnostic
    # needs them to demonstrate *why* pKi is dropped. Everything from Part A.2c onward
    # models `primary_endpoint` alone.
    task_mode: str = "classification"        # "classification" | "regression"
    primary_endpoint: str = "pIC50"          # the one endpoint that gets modelled
    primary_affinity_column: str = "IC50 (nM)"

    # Pre-registered class boundaries on the pAffinity scale. Anchored on pharmacology
    # (10 nM / 1 uM), NOT on quantiles of this particular dataset - a quantile cut makes
    # "active" mean "top third of this snapshot", which is not transferable and shifts
    # when data is added.
    n_activity_classes: int = 2              # 2 = active vs not-active; 3 = + moderate
    active_threshold: float = 8.0            # pIC50 >= 8.0  <=>  IC50 <= 10 nM
    inactive_threshold: float = 6.0          # pIC50 <  6.0  <=>  IC50 >  1 uM (3-class only)
    keep_censored_as_labels: bool = True     # recover '>' rows as inactives, '<' as actives
    class_weight_balanced: bool = True       # class-weighted loss / fitting

    # Exploratory: down-weight over-represented sources (one patent is 26% of the
    # pIC50 labels). Inverse-sqrt-frequency per source tag. Default OFF - this is a
    # modelling choice with no obviously right answer, exposed so its effect is
    # measurable rather than assumed.
    downweight_dominant_source: bool = False

    # ---- Applicability domain ---------------------------------------------------
    # F5's consensus rule let compounds pass while failing the Tanimoto test, which for a
    # congeneric target is the flag with teeth: all 15 F5 top hits were tanimoto_OOD and
    # all 15 passed. With the veto on, failing Tanimoto alone puts a compound out of
    # domain regardless of the other three metrics.
    ad_tanimoto_veto: bool = True
    ad_consensus_min_flags: int = 3

    # ---- Leave-one-cluster-out evaluation ---------------------------------------
    butina_cutoff: float = 0.4               # Tanimoto distance cutoff for clustering
    loco_max_clusters: int = 12              # largest N clusters get their own fold
    loco_min_cluster_size: int = 12          # smaller clusters are pooled into one fold
    loco_random_reference_repeats: int = 10  # random-split repeats, for comparison only

    # ---- Screening metrics -------------------------------------------------------
    enrichment_fractions: Sequence[float] = field(
        default_factory=lambda: [0.01, 0.05, 0.10]
    )
    precision_at_n: Sequence[int] = field(default_factory=lambda: [50, 100, 200])

    def __post_init__(self):
        Path(self.data_dir).mkdir(parents=True, exist_ok=True)
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


CONFIG = Config()
CONFIG

## 2. Part A.1 — ChEMBL pretraining set: PSMA-related targets only

Rather than pretraining on an arbitrary cross-section of ChEMBL, this pulls
bioactivities **only from targets related to PSMA** — controlled by
`CONFIG.chembl_pretrain_target_keywords`. PSMA (GCPII/FOLH1) is a zinc-dependent,
co-catalytic-dinuclear-zinc **metallopeptidase** (MEROPS family M28), and the great
majority of known PSMA inhibitors are built around a handful of zinc-binding groups
(phosphonates, thiols, hydroxamates, ureas) shared with other zinc metallopeptidase
inhibitors. So the "related" pool queried here is: PSMA's own paralog family
(other glutamate carboxypeptidases), other aminopeptidases/carboxypeptidases,
and the classic zinc-metalloprotease drug targets whose inhibitor chemistry PSMA
inhibitors borrow from directly — ACE, neprilysin, ECE-1, thermolysin, MMPs,
insulin-degrading enzyme, meprin — plus carbonic anhydrase, included for its
structurally diverse zinc-binding inhibitor literature even though its fold differs.

This should transfer better than a target-agnostic broad set because the trunk
learns the specific structure-activity vocabulary of zinc-binding-group chemistry,
not generic bioactivity patterns from unrelated target classes. Cached to CSV so
repeated runs don't re-hit the API; if ChEMBL is unreachable, or too few related
targets are found, it falls back first to the old **broad** (target-agnostic) mode
and then to a small synthetic set, so the rest of the notebook always still runs.


In [ ]:
def _synthetic_chembl_pretrain_set(cfg: Config, n: int = 500) -> pd.DataFrame:
    '''Small, clearly-labeled synthetic stand-in used only if ChEMBL can't be reached.'''
    print("!! Using SYNTHETIC ChEMBL pretraining data (network/API unavailable) !!")
    demo_smiles = [
        "CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
        "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
        "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
        "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12",
        "Cc1ccc(cc1)S(=O)(=O)N", "O=C(Nc1ccccc1)c1ccccc1O",
        "CCN(CC)CCOC(=O)c1ccc(N)cc1", "COc1ccc2[nH]c(nc2c1)S(=O)Cc1ncc(C)c(OC)c1C",
    ]
    rng = np.random.RandomState(cfg.random_state)
    rows = []
    for i in range(n):
        base = demo_smiles[i % len(demo_smiles)]
        rows.append({
            "smiles": base,
            "target_chembl_id": f"SYNTH_TARGET_{i % 40}",
            cfg.chembl_pretrain_endpoint_name: float(np.clip(rng.normal(6.5, 1.5), 3.0, 10.0)),
        })
    return pd.DataFrame(rows)


def find_related_chembl_targets(cfg: Config) -> pd.DataFrame:
    '''Searches ChEMBL's target registry for every target whose preferred name matches
    one of `chembl_pretrain_target_keywords`, returning a deduplicated
    (target_chembl_id, pref_name, organism, matched_keyword) table.'''
    from chembl_webresource_client.new_client import new_client

    target_client = new_client.target
    rows = []
    for kw in cfg.chembl_pretrain_target_keywords:
        try:
            hits = target_client.filter(pref_name__icontains=kw, target_type=cfg.chembl_pretrain_target_type) \
                                 .only(["target_chembl_id", "pref_name", "organism"])
            for h in hits:
                rows.append({"target_chembl_id": h.get("target_chembl_id"), "pref_name": h.get("pref_name"),
                             "organism": h.get("organism"), "matched_keyword": kw})
        except Exception as e:
            print(f"  target search failed for keyword '{kw}': {e}")
    if not rows:
        return pd.DataFrame(columns=["target_chembl_id", "pref_name", "organism", "matched_keyword"])
    df = pd.DataFrame(rows).dropna(subset=["target_chembl_id"])
    df = df.drop_duplicates(subset=["target_chembl_id"]).reset_index(drop=True)
    return df


def fetch_activities_for_targets(cfg: Config, target_ids: Sequence[str]) -> pd.DataFrame:
    '''Pulls bioactivities restricted to a specific list of target_chembl_ids, in
    batches (long __in lists are chunked to keep each request well-formed).'''
    from chembl_webresource_client.new_client import new_client

    activity_client = new_client.activity
    rows = []
    per_target_counts: Dict[str, int] = {}
    batch_size = 40
    target_ids = list(target_ids)

    for i in range(0, len(target_ids), batch_size):
        batch = target_ids[i:i + batch_size]
        query = activity_client.filter(
            target_chembl_id__in=batch,
            standard_type__in=list(cfg.chembl_pretrain_standard_types),
            pchembl_value__isnull=False,
        ).only(["canonical_smiles", "target_chembl_id", "pchembl_value"])

        for rec in tqdm(query, desc=f"ChEMBL activities (targets {i+1}-{i+len(batch)}/{len(target_ids)})"):
            smi = rec.get("canonical_smiles")
            tgt = rec.get("target_chembl_id")
            pval = rec.get("pchembl_value")
            if not smi or not tgt or pval is None:
                continue
            if per_target_counts.get(tgt, 0) >= cfg.chembl_pretrain_per_target_cap:
                continue
            per_target_counts[tgt] = per_target_counts.get(tgt, 0) + 1
            rows.append({"smiles": smi, "target_chembl_id": tgt, cfg.chembl_pretrain_endpoint_name: float(pval)})
            if len(rows) >= cfg.chembl_pretrain_max_compounds:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)


def fetch_chembl_pretraining_set_broad(cfg: Config) -> pd.DataFrame:
    '''Original target-agnostic strategy: sample activities across arbitrary human
    targets. Used as the "broad" mode, and as an automatic fallback if domain-targeted
    search doesn't find enough related targets.'''
    from chembl_webresource_client.new_client import new_client

    activity_client = new_client.activity
    rows, seen_targets, per_target_counts = [], set(), {}
    query = activity_client.filter(
        standard_type__in=list(cfg.chembl_pretrain_standard_types),
        target_organism=cfg.chembl_pretrain_organism,
        pchembl_value__isnull=False,
    ).only(["canonical_smiles", "target_chembl_id", "pchembl_value"])

    for rec in tqdm(query, total=cfg.chembl_pretrain_max_compounds * 3, desc="ChEMBL pretraining set (broad)"):
        smi, tgt, pval = rec.get("canonical_smiles"), rec.get("target_chembl_id"), rec.get("pchembl_value")
        if not smi or not tgt or pval is None:
            continue
        if tgt not in per_target_counts and len(seen_targets) >= cfg.chembl_pretrain_max_targets:
            continue
        if per_target_counts.get(tgt, 0) >= 60:
            continue
        seen_targets.add(tgt)
        per_target_counts[tgt] = per_target_counts.get(tgt, 0) + 1
        rows.append({"smiles": smi, "target_chembl_id": tgt, cfg.chembl_pretrain_endpoint_name: float(pval)})
        if len(rows) >= cfg.chembl_pretrain_max_compounds:
            break
    return pd.DataFrame(rows)


def fetch_chembl_pretraining_set(cfg: Config) -> pd.DataFrame:
    cache_path = Path(cfg.chembl_pretrain_cache)
    if cache_path.exists() and not cfg.force_rebuild_chembl_cache:
        print(f"Found cached ChEMBL pretraining set: {cache_path}")
        return pd.read_csv(cache_path)

    try:
        if cfg.chembl_pretrain_mode == "domain_targeted":
            print(f"Searching ChEMBL for targets related to PSMA via {len(cfg.chembl_pretrain_target_keywords)} "
                  f"keyword(s): {list(cfg.chembl_pretrain_target_keywords)}")
            targets_df = find_related_chembl_targets(cfg)
            print(f"Found {len(targets_df):,} matching targets.")
            if len(targets_df):
                print(targets_df[["pref_name", "organism", "matched_keyword"]].to_string(index=False))

            if len(targets_df) < cfg.chembl_pretrain_min_targets_or_fallback_to_broad:
                print(f"Fewer than {cfg.chembl_pretrain_min_targets_or_fallback_to_broad} related targets found "
                      f"-> falling back to 'broad' mode for this run.")
                df = fetch_chembl_pretraining_set_broad(cfg)
            else:
                df = fetch_activities_for_targets(cfg, targets_df["target_chembl_id"].tolist())
                df = df.merge(targets_df[["target_chembl_id", "pref_name", "matched_keyword"]],
                               on="target_chembl_id", how="left")
        else:
            df = fetch_chembl_pretraining_set_broad(cfg)

        if not len(df):
            raise RuntimeError("ChEMBL query returned no usable rows.")

        df = df.drop_duplicates(subset=["smiles"]).reset_index(drop=True)
        print(f"\nFetched {len(df):,} ChEMBL pretraining compounds across "
              f"{df['target_chembl_id'].nunique():,} targets.")
        if "pref_name" in df.columns:
            print("Compounds per target:")
            print(df["pref_name"].value_counts().head(20).to_string())

    except Exception as e:
        print(f"Could not fetch live ChEMBL data ({e}); falling back to a synthetic set.")
        df = _synthetic_chembl_pretrain_set(cfg)

    df.to_csv(cache_path, index=False)
    print(f"Cached ChEMBL pretraining set to {cache_path}")
    return df


pretrain_df = fetch_chembl_pretraining_set(CONFIG)
pretrain_df.head()


## 3. Part A.2 — BindingDB PSMA fine-tuning set (multi-task: pIC50 + pKi)

Same DuckDB-over-the-full-TSV approach as before, but now pulling **both** affinity
columns in `CONFIG.affinity_columns` (IC50 and Ki) and pivoting to a **wide,
multi-task table**: one row per compound with a `pIC50` column, a `pKi` column, and
`NaN` in whichever one a given compound doesn't have a measurement for. Downstream,
the multi-task model trains on both columns at once with a masked loss, so compounds
don't need *both* labels to contribute.

**Kept as a multi-task pull in F6.** Both endpoints are still fetched here even though
only `primary_endpoint` is modelled, because Part A.2b's diagnostic needs both to
demonstrate *why* pKi is dropped — 111 of the dual-measured compounds have pKi derived
from pIC50 by a fixed Cheng-Prusoff factor of 140. Part A.2c re-derives the dataset as a
single-task classification set from the same raw pull.


In [ ]:
def resolve_bindingdb_tsv_url(cfg: Config) -> tuple:
    '''Scrape the BindingDB download page for the CURRENT 'BindingDB_All_<YYYYMM>_tsv.zip'
    filename (BindingDB republishes this monthly, so a hardcoded name goes stale).'''
    download_page = "https://www.bindingdb.org/rwd/bind/chemsearch/marvin/Download.jsp"
    try:
        resp = requests.get(download_page, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resp.raise_for_status()
        m = re.search(r"download_file=(/rwd/bind/downloads/BindingDB_All_\d{6}_tsv\.zip)", resp.text)
        if m:
            file_param = m.group(1)
            zip_name = file_param.rsplit("/", 1)[-1]
            direct_url = f"https://www.bindingdb.org{file_param}"
            print(f"Resolved current BindingDB TSV: {zip_name}")
            return direct_url, zip_name
        print(f"Could not find the current filename on {download_page}; using CONFIG default.")
    except Exception as e:
        print(f"Could not reach {download_page} ({e}); using CONFIG default.")
    return cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name


def _fetch_to_file(url: str, dest: Path) -> str:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; AD-pipeline/1.0)"}
    with requests.get(url, stream=True, timeout=120, headers=headers) as r:
        r.raise_for_status()
        content_type = r.headers.get("content-type", "")
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pbar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                pbar.update(len(chunk))
    return content_type


def _recover_real_download_link(html_path: Path) -> Optional[str]:
    try:
        text = html_path.read_text(errors="ignore")
    except Exception:
        return None
    m = re.search(r'(https://www\.bindingdb\.org/rwd/bind/downloads/[^\s"\x27<>)]+\.zip)', text)
    return m.group(1) if m else None


def download_bindingdb(cfg: Config) -> Optional[Path]:
    '''Download (if needed) and unzip the full BindingDB TSV dump. Returns None (instead of
    raising) if the download can't be completed, so the notebook can fall back to a
    synthetic fine-tuning set and still run end-to-end.'''
    data_dir = Path(cfg.data_dir)
    try:
        if cfg.auto_resolve_bindingdb_url:
            url, zip_name = resolve_bindingdb_tsv_url(cfg)
        else:
            url, zip_name = cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name

        zip_path = data_dir / zip_name
        tsv_path = data_dir / zip_name.replace(".zip", "")

        if tsv_path.exists() and not cfg.force_redownload:
            print(f"Found existing TSV: {tsv_path}")
            return tsv_path

        if not zip_path.exists() or cfg.force_redownload:
            print(f"Downloading {url} ...")
            content_type = _fetch_to_file(url, zip_path)
            print(f"Saved to {zip_path} (content-type: {content_type})")

            if not zipfile.is_zipfile(zip_path):
                real_url = _recover_real_download_link(zip_path)
                if real_url and real_url != url:
                    print(f"Not a zip — retrying with the real link found on the landing page: {real_url}")
                    _fetch_to_file(real_url, zip_path)

        if not zipfile.is_zipfile(zip_path):
            with open(zip_path, "rb") as f:
                head = f.read(300)
            zip_path.unlink(missing_ok=True)
            raise RuntimeError(f"Downloaded file is not a valid zip. First bytes: {head!r}")

        print("Unzipping ...")
        with zipfile.ZipFile(zip_path) as zf:
            member = [m for m in zf.namelist() if m.lower().endswith(".tsv")][0]
            with zf.open(member) as src, open(tsv_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
        print(f"Extracted: {tsv_path}")
        return tsv_path

    except Exception as e:
        print(f"Could not obtain the BindingDB TSV dump ({e}).")
        return None


def query_bindingdb_duckdb_multitask(tsv_path: Path, cfg: Config) -> pd.DataFrame:
    '''Pull one long dataframe with a row per (compound, endpoint) for every column in
    `cfg.affinity_columns`, using DuckDB SQL streaming over the TSV (never materialized
    fully in pandas).'''
    con = duckdb.connect(database=":memory:")
    con.execute("PRAGMA threads=4;")
    con.execute(f'''
        CREATE OR REPLACE VIEW bdb AS
        SELECT * FROM read_csv(
            '{tsv_path.as_posix()}', delim='\t', header=True, quote='"', escape='"',
            strict_mode=False, ignore_errors=True, all_varchar=True
        )
    ''')

    base_where = ['"Ligand SMILES" IS NOT NULL']
    if cfg.target_uniprot_id:
        uniprot_conditions = [
            f'"UniProt (SwissProt) Primary ID of Target Chain {i}" = \'{cfg.target_uniprot_id}\''
            for i in range(1, 6)
        ]
        base_where.append("(" + " OR ".join(uniprot_conditions) + ")")
    if cfg.target_name_like:
        safe = cfg.target_name_like.replace("'", "''")
        base_where.append(f'"Target Name" ILIKE \'%{safe}%\'')
    if cfg.organism_like:
        safe = cfg.organism_like.replace("'", "''")
        base_where.append(f'"Target Source Organism According to Curator or DataSource" ILIKE \'%{safe}%\'')

    frames = []
    for col, endpoint in zip(cfg.affinity_columns, cfg.endpoint_names):
        where_sql = " AND ".join(base_where + [f'"{col}" IS NOT NULL'])
        query = f'''
            SELECT
                "Ligand SMILES"                                    AS smiles,
                "BindingDB Ligand Name"                            AS ligand_name,
                "{col}"                                            AS affinity_raw,
                '{endpoint}'                                       AS endpoint_name
            FROM bdb
            WHERE {where_sql}
        '''
        part = con.execute(query).fetch_df()
        print(f"  {endpoint} ('{col}'): {len(part):,} raw rows.")
        frames.append(part)
    con.close()
    return pd.concat(frames, ignore_index=True)


def _synthetic_bindingdb_finetune_set(cfg: Config, n: int = 250) -> pd.DataFrame:
    print("!! Using SYNTHETIC BindingDB PSMA fine-tuning data (network/download unavailable) !!")
    demo_smiles = [
        "CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
        "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
        "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
        "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12",
        "NC(CCC(=O)O)C(=O)O", "OC(=O)CCC(NC(=O)NC(CCC(=O)O)C(=O)O)C(=O)O",
    ]
    rng = np.random.RandomState(cfg.random_state)
    rows = []
    for i in range(n):
        smi = demo_smiles[i % len(demo_smiles)]
        row = {"smiles": smi, "ligand_name": f"synthetic_{i}"}
        if rng.rand() < 0.75:
            row[cfg.endpoint_names[0]] = float(np.clip(rng.normal(6.8, 1.2), 3, 10))
        if rng.rand() < 0.6:
            row[cfg.endpoint_names[1]] = float(np.clip(rng.normal(6.5, 1.3), 3, 10))
        rows.append(row)
    return pd.DataFrame(rows)


def clean_bindingdb_multitask(df_long: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    '''Coerce to numeric, drop censored (>,<) measurements, compute pAffinity, take the
    per-compound median for each endpoint separately, then pivot wide so every
    compound is one row with (possibly-NaN) pIC50 / pKi columns.'''
    df = df_long.copy()
    df["is_censored"] = df["affinity_raw"].astype(str).str.contains("[<>]", regex=True)
    df = df[~df["is_censored"]]
    df["affinity_nm"] = pd.to_numeric(df["affinity_raw"], errors="coerce")
    df = df.dropna(subset=["affinity_nm", "smiles"])
    df = df[(df["affinity_nm"] >= cfg.min_affinity_nm) & (df["affinity_nm"] <= cfg.max_affinity_nm)]
    df["pAffinity"] = 9 - np.log10(df["affinity_nm"])

    per_endpoint = (
        df.groupby(["smiles", "endpoint_name"], as_index=False)
        .agg({"pAffinity": "median", "ligand_name": "first"})
    )
    wide = per_endpoint.pivot_table(index="smiles", columns="endpoint_name", values="pAffinity")
    names = per_endpoint.groupby("smiles")["ligand_name"].first()
    wide = wide.join(names).reset_index()
    for ep in cfg.endpoint_names:
        if ep not in wide.columns:
            wide[ep] = np.nan
    print(f"Cleaned multi-task fine-tuning set: {len(wide):,} unique compounds "
          + ", ".join(f"({ep}: {wide[ep].notna().sum():,})" for ep in cfg.endpoint_names))
    return wide.reset_index(drop=True)


# --- run it -------------------------------------------------------------------
tsv_path = download_bindingdb(CONFIG)
if tsv_path is not None:
    raw_bdb_df = query_bindingdb_duckdb_multitask(tsv_path, CONFIG)
    train_df = clean_bindingdb_multitask(raw_bdb_df, CONFIG)
else:
    train_df = _synthetic_bindingdb_finetune_set(CONFIG)

train_df.head()


## 3b. Part A.2b — Where should the activity-class cuts go?

A read-only diagnostic, added to settle the threshold question before any switch from
numeric prediction to activity classes. It defines nothing the rest of the pipeline
consumes, so it can be run or skipped freely.

It needs `raw_bdb_df` and `train_df`, both created by the cell above — `raw_bdb_df`
is the **pre-cleaning** long frame, so it still contains the censored (`>` / `<`)
measurements that `clean_bindingdb_multitask` discards. That is what makes the
censored-row audit possible without re-downloading anything. If the kernel has been
restarted, re-run the cell above first (it reuses the cached TSV).

Four questions, in order:

1. **Shape.** Percentiles, skew, kurtosis, a Gaussian-mixture BIC comparison for
   1 / 2 / 3 components, and a kernel-density scan for the *valleys* — the emptiest
   places to put a boundary, where a cut splits the fewest near-identical compounds.
2. **Censored-row audit.** How many `>` and `<` rows were dropped per endpoint, where
   their bounds sit (a `>1 µM` is a far weaker claim than a `>100 µM`), and how many
   distinct compounds they would add back as confident inactives or actives. This is
   the single number that decides three-class versus binary.
3. **Class counts** under candidate schemes side by side: field-standard
   (100 nM / 10 µM), PSMA-tightened (10 nM / 1 µM), PSMA-strict (1 nM / 1 µM), plus
   observed tertiles and the density valleys for comparison — each shown before and
   after the censored data is folded in.
4. **Boundary noise.** For compounds measured both ways, how often pIC50 and pKi land
   in *different* classes. That disagreement is assay noise on the same molecule, so it
   is a hard ceiling on classifier accuracy. A scheme below roughly 80% agreement has
   boundaries cutting through noise rather than signal.

Note on interpretation: anchor the thresholds on pharmacology and use the distribution
to *check* them — that each class has enough members, and that the cut does not land on
a density peak. Choosing tertiles because they balance the classes makes "active" mean
"top third of this particular snapshot", which is not transferable and shifts the moment
ChEMBL data or decoys are added.


In [ ]:
# =====================================================================================
# Part A.2b — Label-distribution diagnostic: where should the activity-class cuts go?
#
# Drop this in as a new cell immediately AFTER cell 9 (the BindingDB pull), which is
# where `raw_bdb_df` and `train_df` come into existence. It is read-only: it adds no
# variables the rest of the pipeline depends on and changes nothing.
#
# It answers four questions:
#   1. What shape is the label distribution really? (percentiles, skew, bimodality)
#   2. How many censored (">" / "<") rows were dropped, and what would they add back
#      as confident inactives / actives?
#   3. How many compounds land in each class under candidate threshold schemes?
#   4. How noisy is the labelling at the boundaries, measured by pIC50-vs-pKi
#      disagreement on compounds that have both?
# =====================================================================================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

EPS = list(CONFIG.endpoint_names)          # ['pIC50', 'pKi']
_COLS = dict(zip(CONFIG.endpoint_names, CONFIG.affinity_columns))

print("=" * 86)
print("1. SHAPE OF THE LABEL DISTRIBUTION")
print("=" * 86)

_shape_rows = []
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) < 10:
        continue
    q = np.percentile(v, [1, 5, 10, 25, 50, 75, 90, 95, 99])
    _shape_rows.append({
        "endpoint": ep, "n": len(v),
        "mean": v.mean(), "sd": v.std(ddof=0),
        "p1": q[0], "p5": q[1], "p10": q[2], "p25": q[3], "p50": q[4],
        "p75": q[5], "p90": q[6], "p95": q[7], "p99": q[8],
        "skew": stats.skew(v), "excess_kurt": stats.kurtosis(v),
        "min": v.min(), "max": v.max(),
    })
shape_df = pd.DataFrame(_shape_rows).set_index("endpoint").round(3)
display(shape_df)

# --- is it one mode or several? -------------------------------------------------------
# A kernel-density scan: local minima of the density between the 5th and 95th percentile
# are the natural "valleys", i.e. the least-populated places to put a cut.
def density_valleys(v, n_grid=512, lo_pct=5, hi_pct=95):
    kde = stats.gaussian_kde(v)
    lo, hi = np.percentile(v, [lo_pct, hi_pct])
    grid = np.linspace(lo, hi, n_grid)
    dens = kde(grid)
    # interior local minima
    is_min = (dens[1:-1] < dens[:-2]) & (dens[1:-1] < dens[2:])
    valleys = grid[1:-1][is_min]
    depths = dens[1:-1][is_min]
    order = np.argsort(depths)              # shallowest density = emptiest = best cut
    return grid, dens, valleys[order], depths[order]

print("\nBimodality check and natural density valleys")
print("-" * 86)
valley_store = {}
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) < 30:
        print(f"  {ep}: too few labelled rows for a density scan.")
        continue
    grid, dens, valleys, depths = density_valleys(v)
    valley_store[ep] = (grid, dens, valleys)
    # Crude unimodality signal: compare the best single Gaussian to a 2-component mix.
    try:
        from sklearn.mixture import GaussianMixture
        X = v.reshape(-1, 1)
        bic1 = GaussianMixture(1, random_state=0).fit(X).bic(X)
        bic2 = GaussianMixture(2, random_state=0).fit(X).bic(X)
        bic3 = GaussianMixture(3, random_state=0).fit(X).bic(X)
        best = {1: bic1, 2: bic2, 3: bic3}
        n_modes = min(best, key=best.get)
        verdict = (f"{n_modes}-component mixture fits best "
                   f"(BIC 1/2/3 = {bic1:.0f} / {bic2:.0f} / {bic3:.0f})")
    except Exception as e:
        verdict = f"mixture fit unavailable ({e})"
    vstr = ", ".join(f"{x:.2f}" for x in valleys[:4]) if len(valleys) else "none found"
    print(f"  {ep}: {verdict}")
    print(f"         emptiest cut points (shallowest density first): {vstr}")
    print(f"         -> a threshold placed in a valley splits fewer near-identical "
          f"compounds across the boundary")

print("\n" + "=" * 86)
print("2. CENSORED-ROW AUDIT  (the data classification would recover)")
print("=" * 86)

_cens_rows = []
_recovered = {}          # endpoint -> DataFrame of censored measurements
for ep in EPS:
    col = _COLS[ep]
    sub = raw_bdb_df[raw_bdb_df["endpoint_name"] == ep].copy()
    raw = sub["affinity_raw"].astype(str)
    sub["qualifier"] = raw.str.extract(r"^\s*([<>]=?)")[0]
    sub["value_nm"] = pd.to_numeric(raw.str.replace(r"^\s*[<>]=?\s*", "", regex=True),
                                    errors="coerce")
    cens = sub[sub["qualifier"].notna() & sub["value_nm"].notna()].copy()
    cens["pAffinity_bound"] = 9 - np.log10(cens["value_nm"])
    _recovered[ep] = cens

    greater = cens[cens["qualifier"].str.startswith(">")]
    lesser = cens[cens["qualifier"].str.startswith("<")]
    _cens_rows.append({
        "endpoint": ep,
        "raw_rows": len(sub),
        "censored_rows": len(cens),
        "gt_rows_INACTIVE": len(greater),
        "lt_rows_ACTIVE": len(lesser),
        "gt_distinct_cpds": greater["smiles"].nunique(),
        "lt_distinct_cpds": lesser["smiles"].nunique(),
        "numeric_cpds_now": int(train_df[ep].notna().sum()),
    })

cens_df = pd.DataFrame(_cens_rows).set_index("endpoint")
display(cens_df)

print("Where the '>' bounds sit (these become inactives; a '>1 uM' is a much weaker")
print("statement than a '>100 uM', so they are not interchangeable):")
for ep in EPS:
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g) == 0:
        print(f"  {ep}: no '>' rows.")
        continue
    vc = g["value_nm"].value_counts().sort_index()
    print(f"  {ep}:")
    for val, n in vc.items():
        print(f"      > {val:>12,.1f} nM  (pAffinity < {9 - np.log10(val):.2f})   n = {n}")

# How many of the '>' compounds are genuinely NEW, vs already present with a number?
print("\nOverlap with the compounds already in train_df:")
for ep in EPS:
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g) == 0:
        continue
    new = set(g["smiles"]) - set(train_df["smiles"])
    print(f"  {ep}: {g['smiles'].nunique()} distinct '>' compounds, "
          f"of which {len(new)} are not in the current training set at all")
print("  (Note: matching is on raw SMILES strings, as the pipeline does, so this is an")
print("   under-count of true overlap - standardisation would merge more. Dedup on")
print("   InChIKey after standardisation before trusting these numbers.)")

print("\n" + "=" * 86)
print("3. CLASS COUNTS UNDER CANDIDATE THRESHOLD SCHEMES")
print("=" * 86)

# (name, active_cut, inactive_cut) on the pAffinity scale; class is
#   ACTIVE    if  p >= active_cut
#   INACTIVE  if  p <  inactive_cut
#   MODERATE  otherwise
_p = lambda nm: 9 - np.log10(nm)

SCHEMES = {
    "field-standard (100 nM / 10 uM)": (_p(100), _p(10_000)),
    "PSMA-tightened (10 nM / 1 uM)":   (_p(10),  _p(1_000)),
    "PSMA-strict (1 nM / 1 uM)":       (_p(1),   _p(1_000)),
}
# add tertiles of the observed numeric distribution, per endpoint, for comparison only
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) > 30:
        lo, hi = np.percentile(v, [33.3, 66.7])
        SCHEMES[f"tertiles of observed {ep} ({hi:.2f} / {lo:.2f})"] = (hi, lo)
# add the emptiest density valleys, if two were found
for ep in EPS:
    if ep in valley_store and len(valley_store[ep][2]) >= 2:
        two = sorted(valley_store[ep][2][:2])
        SCHEMES[f"density valleys of {ep} ({two[1]:.2f} / {two[0]:.2f})"] = (two[1], two[0])


def classify(p, active_cut, inactive_cut):
    if not np.isfinite(p):
        return np.nan
    if p >= active_cut:
        return "active"
    if p < inactive_cut:
        return "inactive"
    return "moderate"


rows = []
for name, (a_cut, i_cut) in SCHEMES.items():
    for ep in EPS:
        v = train_df[ep].dropna()
        cls = v.map(lambda p: classify(p, a_cut, i_cut))
        counts = cls.value_counts()
        # censored '>' rows become inactives, '<' rows become actives
        g = _recovered[ep]
        add_inact = g[g["qualifier"].str.startswith(">")]
        # only count a '>' row as inactive if its bound actually implies inactivity
        add_inact = add_inact[add_inact["pAffinity_bound"] <= i_cut]
        add_act = g[g["qualifier"].str.startswith("<")]
        add_act = add_act[add_act["pAffinity_bound"] >= a_cut]
        rows.append({
            "scheme": name, "endpoint": ep,
            "active": int(counts.get("active", 0)),
            "moderate": int(counts.get("moderate", 0)),
            "inactive": int(counts.get("inactive", 0)),
            "add_inact": add_inact["smiles"].nunique(),
            "add_act": add_act["smiles"].nunique(),
            "inact_total": int(counts.get("inactive", 0)) + add_inact["smiles"].nunique(),
        })

class_counts = pd.DataFrame(rows).set_index(["scheme", "endpoint"])
display(class_counts)

print("Reading this table:")
print("  * 'inact_total' is the figure that decides 3-class vs binary. Under ~80")
print("    compounds and three classes will not train reliably - go binary")
print("    (active vs not-active) instead.")
print("  * A scheme that puts >70% of compounds in one class will need class weights")
print("    or stratified sampling whichever way you go.")
print("  * Prefer a pharmacologically meaningful cut that happens to sit near a density")
print("    valley. Do NOT pick tertiles because they balance the classes - that makes")
print("    'active' mean 'top third of this snapshot', which moves when data is added.")

print("\n" + "=" * 86)
print("4. BOUNDARY NOISE: do pIC50 and pKi agree on the class?")
print("=" * 86)
print("Compounds measured both ways should land in the same class. Where they do not,")
print("the disagreement is assay noise, and it bounds how good any classifier can be.\n")

both = train_df[train_df[EPS[0]].notna() & train_df[EPS[1]].notna()]
print(f"Compounds with both {EPS[0]} and {EPS[1]}: {len(both)}")
if len(both) >= 10:
    d = (both[EPS[0]] - both[EPS[1]]).abs()
    print(f"  |{EPS[0]} - {EPS[1]}|:  median {d.median():.2f}, mean {d.mean():.2f}, "
          f"90th pct {d.quantile(0.9):.2f}, max {d.max():.2f} log units")
    print(f"  fraction differing by >0.5 log: {100 * (d > 0.5).mean():.1f}%")
    print(f"  fraction differing by >1.0 log: {100 * (d > 1.0).mean():.1f}%")
    print(f"  Pearson r between the two endpoints: {both[EPS[0]].corr(both[EPS[1]]):.3f}\n")
    for name, (a_cut, i_cut) in SCHEMES.items():
        c0 = both[EPS[0]].map(lambda p: classify(p, a_cut, i_cut))
        c1 = both[EPS[1]].map(lambda p: classify(p, a_cut, i_cut))
        agree = (c0 == c1).mean()
        try:
            from sklearn.metrics import cohen_kappa_score
            kappa = cohen_kappa_score(c0, c1)
            kstr = f", kappa = {kappa:.3f}"
        except Exception:
            kstr = ""
        print(f"  {name:<46s} agreement = {100 * agree:5.1f}%{kstr}")
    print("\n  Agreement here is a CEILING on classifier accuracy: the same molecule,")
    print("  measured twice, disagrees this often. If a scheme falls below ~80%, its")
    print("  boundaries are cutting through the noise rather than through the signal.")
else:
    print("  Too few dual-measured compounds to assess boundary noise.")

print("\n" + "=" * 86)
print("5. PLOTS")
print("=" * 86)

fig, axes = plt.subplots(1, len(EPS), figsize=(7.5 * len(EPS), 4.8))
axes = np.atleast_1d(axes)
_cut_colors = plt.get_cmap("tab10")
for ax, ep in zip(axes, EPS):
    v = train_df[ep].dropna().values
    ax.hist(v, bins=40, color="#9aa0a6", edgecolor="white", linewidth=0.5,
            alpha=0.85, label=f"{ep} (n={len(v)})")
    if ep in valley_store:
        grid, dens, valleys = valley_store[ep]
        ax2 = ax.twinx()
        ax2.plot(grid, dens, color="#1a1a1a", linewidth=1.6, label="density")
        ax2.set_yticks([])
        for x in valleys[:3]:
            ax2.axvline(x, color="#1a1a1a", linestyle=":", linewidth=1.2, alpha=0.7)
    for ci, (name, (a_cut, i_cut)) in enumerate(list(SCHEMES.items())[:3]):
        col = _cut_colors(ci)
        ax.axvline(a_cut, color=col, linestyle="--", linewidth=1.8,
                   label=f"{name.split(' (')[0]}: active")
        ax.axvline(i_cut, color=col, linestyle="-.", linewidth=1.4, alpha=0.8)
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g):
        ax.scatter(g["pAffinity_bound"], np.full(len(g), ax.get_ylim()[1] * 0.04),
                   marker="|", s=180, color="#d1495b", alpha=0.6,
                   label=f"censored '>' bounds (n={len(g)})")
    ax.set_xlabel(ep)
    ax.set_ylabel("compounds")
    ax.set_title(f"{ep}: distribution, density valleys (dotted) and candidate cuts")
    ax.legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

print("Dotted vertical lines are density valleys - the emptiest places to cut.")
print("Red ticks are the censored '>' bounds that classification would recover as")
print("inactives; where they pile up tells you where the real inactive mass sits.")

## 3c. Part A.2c — Single-task classification dataset

Acting on Part A.2b. Three things happen here.

**pKi is dropped.** Part A.2b showed that 111 of the 169 dual-measured compounds have
pKi − pIC50 pinned at 2.145 log units — a ratio of exactly 140 — and that all of them
come from US20250213705. Those Ki values are Cheng–Prusoff conversions of the IC50
values, not measurements. A second block of 28 shares a ratio of 5.0 from another source.
So 83% of the paired data is arithmetically linked, the second task was re-feeding the
same numbers under a different name, and the masked loss was double-weighting one
patent's congeneric series. The derived-value detector below re-reports this on whatever
data is present, so it stays visible on a future re-run with a newer BindingDB release.

**The censored measurements come back.** `clean_bindingdb_multitask` discards every
`>`/`<` row because regression cannot fit a bound. For a class label a bound is often
sufficient: `IC50 > 100 µM` proves the compound is not active at any sensible threshold.
A `>` row is kept as an inactive when its bound lies at or below `active_threshold`, and
a `<` row as an active when its bound lies at or above it; bounds that straddle the
threshold are uninformative and dropped. This is the only change in the pipeline that
adds genuinely new information rather than rearranging what was already there — though it
is smaller than hoped, ~32 compounds, since most of the 544 → 452 drop was duplicate
merging rather than censoring.

**Every compound gets a source tag**, parsed from its BindingDB ligand name (patent
numbers like `US20250213705`, or `CHEMBL…`). This is what makes the over-representation
visible, drives the optional source down-weighting, and lets the leave-one-cluster-out
report in Part E.2 name the family it is holding out.

The resulting `train_df` has one row per compound with a `y` column (1 = active), a
`label_source` column (`numeric` or `censored`) and a `source_tag`. Featurization and
everything downstream consume it unchanged.

In [ ]:
# --- Build the single-task classification dataset from the raw BindingDB pull ---------
import re

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Tags that identify one filing/paper, as opposed to catch-all buckets. Only a specific
# tag means "one source": CHEMBL covers thousands of unrelated depositions.
GENERIC_SOURCE_TAGS = {"CHEMBL", "other", "unknown"}

PRIMARY = CONFIG.primary_endpoint
PRIMARY_COL = CONFIG.primary_affinity_column
A_CUT = CONFIG.active_threshold
I_CUT = CONFIG.inactive_threshold
N_CLASSES = CONFIG.n_activity_classes

print(f"Target: {PRIMARY} from '{PRIMARY_COL}'")
print(f"Pre-registered cuts: active >= {A_CUT:.2f} "
      f"(IC50 <= {10 ** (9 - A_CUT):.3g} nM)", end="")
if N_CLASSES == 3:
    print(f", inactive < {I_CUT:.2f} (IC50 > {10 ** (9 - I_CUT):.3g} nM)")
else:
    print("  [binary]")


# =====================================================================================
# 1. Derived-value detector: are the two endpoints independent measurements?
# =====================================================================================
def detect_derived_endpoint(df_wide, ep_a, ep_b, min_cluster=5, tol=0.02):
    """Flags blocks of compounds whose two endpoints differ by a FIXED amount.

    A constant offset shared by many compounds is the signature of one endpoint being
    arithmetically derived from the other (e.g. Cheng-Prusoff Ki = IC50 / (1 + [S]/Km)),
    not of two independent assays. Returns (report_df, mask_of_derived_rows).
    """
    if ep_a not in df_wide.columns or ep_b not in df_wide.columns:
        return pd.DataFrame(), pd.Series(dtype=bool)
    both = df_wide[df_wide[ep_a].notna() & df_wide[ep_b].notna()].copy()
    if len(both) < min_cluster:
        return pd.DataFrame(), pd.Series(False, index=df_wide.index)
    both["_diff"] = both[ep_b] - both[ep_a]
    rounded = both["_diff"].round(2)
    rows, derived_idx = [], []
    for val, n in rounded.value_counts().items():
        if n < min_cluster:
            continue
        sel = both[(both["_diff"] - val).abs() <= tol]
        if len(sel) < min_cluster:
            continue
        srcs = sel["source_tag"].value_counts() if "source_tag" in sel else pd.Series(dtype=int)
        rows.append({
            "offset": round(float(sel['_diff'].mean()), 4),
            "implied_ratio": round(float(10 ** sel["_diff"].mean()), 2),
            "n_compounds": len(sel),
            "offset_sd": round(float(sel["_diff"].std(ddof=0)), 5),
            "dominant_source": srcs.index[0] if len(srcs) else "?",
            "source_share": round(float(srcs.iloc[0] / len(sel)), 3) if len(srcs) else np.nan,
        })
        derived_idx.extend(sel.index.tolist())
    mask = pd.Series(False, index=df_wide.index)
    mask.loc[sorted(set(derived_idx))] = True
    return pd.DataFrame(rows).sort_values("n_compounds", ascending=False), mask


# =====================================================================================
# 2. Source tags
# =====================================================================================
_PATENT_RE = re.compile(r"\b((?:US|WO|EP|CN|JP|KR)\s?\d{6,})", re.I)
_CHEMBL_RE = re.compile(r"\b(CHEMBL\d+)", re.I)


def source_tag(name) -> str:
    """A coarse provenance label parsed from the BindingDB ligand name. Patent numbers
    identify single-filing blocks, which is what over-representation looks like here."""
    if not isinstance(name, str) or not name.strip():
        return "unknown"
    m = _PATENT_RE.search(name)
    if m:
        return m.group(1).replace(" ", "").upper()
    if _CHEMBL_RE.search(name):
        return "CHEMBL"
    return "other"


# attach tags to the multi-task frame so the detector can report provenance
train_df_mt = train_df.copy()
train_df_mt["source_tag"] = train_df_mt["ligand_name"].map(source_tag)

derived_report, derived_mask = detect_derived_endpoint(
    train_df_mt, CONFIG.endpoint_names[0], CONFIG.endpoint_names[1]
)
print("\n" + "=" * 86)
print("DERIVED-VALUE DETECTOR — fixed-offset blocks between the two endpoints")
print("=" * 86)
if len(derived_report):
    display(derived_report)
    n_pairs = int((train_df_mt[CONFIG.endpoint_names[0]].notna()
                   & train_df_mt[CONFIG.endpoint_names[1]].notna()).sum())
    print(f"{int(derived_mask.sum())} of {n_pairs} dual-measured compounds "
          f"({100 * derived_mask.sum() / max(n_pairs, 1):.0f}%) sit in a fixed-offset block.")
    print("A constant offset shared across many compounds from one source means the second")
    print("endpoint was computed from the first, not measured independently. This is the")
    print(f"reason the pipeline models {PRIMARY} alone from here on.")
else:
    print("No fixed-offset blocks found. The two endpoints look independently measured;")
    print("a multi-task formulation would be defensible on this data.")

print("\nSource composition of the full multi-task set:")
_src = train_df_mt["source_tag"].value_counts()
for tag, n in _src.head(8).items():
    print(f"  {tag:>16s}: {n:4d} compounds ({100 * n / len(train_df_mt):5.1f}%)")


# =====================================================================================
# 3. Re-derive the primary endpoint, keeping the censored rows
# =====================================================================================
_sub = raw_bdb_df[raw_bdb_df["endpoint_name"] == PRIMARY].copy()
_raw = _sub["affinity_raw"].astype(str)
_sub["qualifier"] = _raw.str.extract(r"^\s*([<>]=?)")[0]
_sub["value_nm"] = pd.to_numeric(_raw.str.replace(r"^\s*[<>]=?\s*", "", regex=True),
                                 errors="coerce")
_sub = _sub.dropna(subset=["value_nm", "smiles"])
_sub = _sub[(_sub["value_nm"] >= CONFIG.min_affinity_nm)
            & (_sub["value_nm"] <= CONFIG.max_affinity_nm)]
_sub["pAff"] = 9 - np.log10(_sub["value_nm"])

_numeric = _sub[_sub["qualifier"].isna()]
_gt = _sub[_sub["qualifier"].notna() & _sub["qualifier"].str.startswith(">")]
_lt = _sub[_sub["qualifier"].notna() & _sub["qualifier"].str.startswith("<")]

# numeric: per-compound median, as before
num_agg = (_numeric.groupby("smiles", as_index=False)
           .agg(pAff=("pAff", "median"), ligand_name=("ligand_name", "first"),
                n_measurements=("pAff", "size")))

# '>' rows: pAffinity is bounded ABOVE. Several bounds must all hold, so take the min
#           (the most informative statement).
gt_agg = (_gt.groupby("smiles", as_index=False)
          .agg(p_upper=("pAff", "min"), ligand_name=("ligand_name", "first")))
# '<' rows: pAffinity is bounded BELOW -> take the max.
lt_agg = (_lt.groupby("smiles", as_index=False)
          .agg(p_lower=("pAff", "max"), ligand_name=("ligand_name", "first")))

print("\n" + "=" * 86)
print(f"RE-DERIVING {PRIMARY} WITH CENSORED ROWS RETAINED")
print("=" * 86)
print(f"  numeric rows           : {len(_numeric):5d}  ->  {len(num_agg):4d} compounds")
print(f"  '>' (upper-bound) rows : {len(_gt):5d}  ->  {len(gt_agg):4d} compounds")
print(f"  '<' (lower-bound) rows : {len(_lt):5d}  ->  {len(lt_agg):4d} compounds")

records = {}
for r in num_agg.itertuples(index=False):
    records[r.smiles] = {"smiles": r.smiles, "ligand_name": r.ligand_name,
                         PRIMARY: float(r.pAff), "p_upper": np.nan, "p_lower": np.nan,
                         "label_source": "numeric", "n_measurements": int(r.n_measurements)}

if CONFIG.keep_censored_as_labels:
    n_new_gt = n_amb_gt = 0
    for r in gt_agg.itertuples(index=False):
        if r.smiles in records:                      # a real number beats a bound
            records[r.smiles]["p_upper"] = float(r.p_upper)
            continue
        if r.p_upper <= A_CUT:                       # provably not active
            records[r.smiles] = {"smiles": r.smiles, "ligand_name": r.ligand_name,
                                 PRIMARY: np.nan, "p_upper": float(r.p_upper),
                                 "p_lower": np.nan, "label_source": "censored",
                                 "n_measurements": 0}
            n_new_gt += 1
        else:
            n_amb_gt += 1
    n_new_lt = n_amb_lt = 0
    for r in lt_agg.itertuples(index=False):
        if r.smiles in records:
            records[r.smiles]["p_lower"] = float(r.p_lower)
            continue
        if r.p_lower >= A_CUT:                       # provably active
            records[r.smiles] = {"smiles": r.smiles, "ligand_name": r.ligand_name,
                                 PRIMARY: np.nan, "p_upper": np.nan,
                                 "p_lower": float(r.p_lower), "label_source": "censored",
                                 "n_measurements": 0}
            n_new_lt += 1
        else:
            n_amb_lt += 1
    print(f"  recovered as INACTIVE  : {n_new_gt:4d} compounds "
          f"({n_amb_gt} '>' bounds straddled the {A_CUT:.1f} cut and were dropped)")
    print(f"  recovered as ACTIVE    : {n_new_lt:4d} compounds "
          f"({n_amb_lt} '<' bounds straddled the cut and were dropped)")
else:
    print("  CONFIG.keep_censored_as_labels is False - censored rows discarded (F5 behaviour)")

cls_df = pd.DataFrame(list(records.values()))
cls_df["source_tag"] = cls_df["ligand_name"].map(source_tag)


# =====================================================================================
# 4. Assign the class label
# =====================================================================================
def assign_class(row) -> float:
    p = row[PRIMARY]
    if np.isfinite(p):
        if N_CLASSES == 2:
            return 1.0 if p >= A_CUT else 0.0
        return 2.0 if p >= A_CUT else (0.0 if p < I_CUT else 1.0)
    pu, pl = row["p_upper"], row["p_lower"]
    if np.isfinite(pu):                      # pAffinity < pu
        if N_CLASSES == 2:
            return 0.0 if pu <= A_CUT else np.nan
        if pu <= I_CUT:
            return 0.0
        return 1.0 if pu <= A_CUT else np.nan
    if np.isfinite(pl):                      # pAffinity > pl
        if N_CLASSES == 2:
            return 1.0 if pl >= A_CUT else np.nan
        if pl >= A_CUT:
            return 2.0
        return 1.0 if pl >= I_CUT else np.nan
    return np.nan


cls_df["y"] = cls_df.apply(assign_class, axis=1)
cls_df = cls_df[cls_df["y"].notna()].reset_index(drop=True)
cls_df["y"] = cls_df["y"].astype(int)

CLASS_NAMES = (["not-active", "active"] if N_CLASSES == 2
               else ["inactive", "moderate", "active"])
POSITIVE_CLASS = N_CLASSES - 1          # "active" is always the last/highest class

print("\n" + "=" * 86)
print("CLASS COMPOSITION")
print("=" * 86)
_comp = (cls_df.groupby(["y", "label_source"]).size()
         .unstack(fill_value=0))
_comp.index = [CLASS_NAMES[i] for i in _comp.index]
_comp["TOTAL"] = _comp.sum(axis=1)
_comp["share"] = (_comp["TOTAL"] / len(cls_df)).round(3)
display(_comp)
print(f"Total modelled compounds: {len(cls_df):,}  "
      f"(base rate for '{CLASS_NAMES[POSITIVE_CLASS]}' = "
      f"{100 * (cls_df['y'] == POSITIVE_CLASS).mean():.1f}%)")

print("\nClass composition by source (top sources):")
_by_src = (cls_df.groupby(["source_tag", "y"]).size().unstack(fill_value=0))
_by_src.columns = [CLASS_NAMES[c] for c in _by_src.columns]
_by_src["TOTAL"] = _by_src.sum(axis=1)
display(_by_src.sort_values("TOTAL", ascending=False).head(8))

_dom = cls_df["source_tag"].value_counts()
_specific = _dom[~_dom.index.isin(GENERIC_SOURCE_TAGS)]
if len(_specific) and _specific.iloc[0] / len(cls_df) > 0.15:
    print(f"!! '{_specific.index[0]}' — a single filing — supplies {_specific.iloc[0]} of "
          f"{len(cls_df)} compounds ({100 * _specific.iloc[0] / len(cls_df):.0f}%).")
    print("   One source at this share means one congeneric series is a large fraction of")
    print("   everything the model can learn. Part E.2's leave-one-cluster-out shows how")
    print("   much of its apparent competence depends on it, and")
    print("   CONFIG.downweight_dominant_source exposes the alternative.")
elif len(_dom):
    print(f"No single filing exceeds 15% of the set (largest specific source: "
          f"{_specific.index[0] if len(_specific) else 'none'}"
          f"{f' at {100 * _specific.iloc[0] / len(cls_df):.0f}%' if len(_specific) else ''}).")

# --- hand over to the rest of the pipeline unchanged ---------------------------------
train_df = cls_df[["smiles", "ligand_name", PRIMARY, "p_upper", "p_lower",
                   "label_source", "source_tag", "y", "n_measurements"]].copy()
print(f"\n`train_df` is now the single-task classification set: {train_df.shape}")
train_df.head()

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully inside Colab.")
except ModuleNotFoundError:
    print("Running on a local environment; skipping Google Drive mount as it is not needed.")

## 4. Part A.3 — Curated screening library

Unchanged from the original pipeline: the compounds you want predictions +
applicability-domain flags for. Supports CSV (with a SMILES column) or SDF.


In [ ]:
def load_screening_library(cfg: Config) -> pd.DataFrame:
    # Sanitize path: if it starts with an absolute root slash like '/Enamine_screen_library.sdf',
    # convert it to a local relative path './Enamine_screen_library.sdf' to avoid Windows permission errors.
    raw_path = cfg.screening_library_path
    if raw_path.startswith("/") or raw_path.startswith("\\"):
        raw_path = "." + raw_path

    path = Path(raw_path)
    if not path.exists():
        print(f"Screening library not found at {path}. Generating a demo screening library locally.")
        path.parent.mkdir(parents=True, exist_ok=True)
        demo = pd.DataFrame({
            "compound_id": [f"demo_{i}" for i in range(8)],
            "SMILES": [
                "CCOc1ccc2nc(S(N)(=O)=O)sc2c1",
                "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
                "O=C(O)c1ccccc1O",
                "CC(=O)Oc1ccccc1C(=O)O",
                "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
                "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
                "CN1CCC[C@H]1c1cccnc1",
                "CC(C)NCC(O)COc1cccc2ccccc12"
            ]
        })
        demo.to_csv(path, index=False)
        print(f"Demo screening library saved to: {path}")

    if path.suffix.lower() in (".sdf", ".sd"):
        suppl = Chem.SDMolSupplier(str(path))
        rows = []
        for i, mol in enumerate(suppl):
            if mol is None:
                continue
            smi = Chem.MolToSmiles(mol)
            cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"SDF_mol_{i}"
            rows.append({"smiles": smi, "compound_id": cid})
        return pd.DataFrame(rows)
    else:
        df = pd.read_csv(path)
        df = df.rename(columns={cfg.screening_smiles_col: "smiles"})
        if cfg.screening_id_col and cfg.screening_id_col in df.columns:
            df = df.rename(columns={cfg.screening_id_col: "compound_id"})
        elif "compound_id" not in df.columns:
            df["compound_id"] = [f"ROW_{i}" for i in range(len(df))]
        return df[["smiles", "compound_id"]]

screen_df = load_screening_library(CONFIG)
screen_df.head()

## 5. Part B.1 — Featurization

Same RDKit standardization + Morgan-fingerprint + descriptor wrapper as the original
pipeline, now applied to **three** sets: the broad ChEMBL pretraining set, the PSMA
BindingDB fine-tuning set, and the screening library. Using the identical featurizer
everywhere is what makes the pretrained trunk's weights meaningful once loaded into
the fine-tuning model — the input feature space has to match.


In [ ]:
class RDKitFeaturizer:
    '''Reusable RDKit wrapper: standardization + fingerprints + descriptors.

    Uses RDKit's full 2D descriptor catalog (~200 physicochemical/topological/graph
    descriptors from `Descriptors._descList`, in the spirit of AutoQSAR's ~500-descriptor
    pool[1]) rather than a small hand-picked subset. The redundancy this introduces is
    dealt with explicitly in Part B.1b below, following AutoQSAR's own near-constant +
    correlation-clustering filter, rather than left for the model to sort out.

    [1] Dixon et al., "AutoQSAR: an automated machine learning tool for best-practice
        QSAR modeling," Future Med. Chem. 2016.
    '''

    DESCRIPTOR_NAMES = [name for name, _ in Descriptors._descList]

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._normalizer = rdMolStandardize.Normalizer()
        self._lfc = rdMolStandardize.LargestFragmentChooser()
        self._uncharger = rdMolStandardize.Uncharger()
        self._desc_fns = dict(Descriptors._descList)

    def standardize(self, smiles: str) -> Optional[Chem.Mol]:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        try:
            mol = self._normalizer.normalize(mol)
            mol = self._lfc.choose(mol)
            mol = self._uncharger.uncharge(mol)
            Chem.SanitizeMol(mol)
        except Exception:
            return None
        return mol

    def morgan_fp(self, mol: Chem.Mol):
        return AllChem.GetMorganFingerprintAsBitVect(mol, self.cfg.morgan_radius, nBits=self.cfg.morgan_n_bits)

    def morgan_fp_with_bit_info(self, mol: Chem.Mol):
        '''Same fingerprint as `morgan_fp`, but also returns RDKit's bitInfo dict
        (bit -> [(center_atom_idx, radius), ...]), needed to map bits back to atoms
        for the interpretability analysis in Part E.6.'''
        bit_info: Dict[int, list] = {}
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, self.cfg.morgan_radius, nBits=self.cfg.morgan_n_bits, bitInfo=bit_info
        )
        return fp, bit_info

    def descriptors(self, mol: Chem.Mol) -> np.ndarray:
        vals = []
        for name in self.DESCRIPTOR_NAMES:
            fn = self._desc_fns.get(name)
            try:
                vals.append(fn(mol) if fn else np.nan)
            except Exception:
                vals.append(np.nan)
        return np.array(vals, dtype=float)

    def featurize_dataframe(self, df: pd.DataFrame, smiles_col: str = "smiles") -> dict:
        mols, fps, desc_rows, keep_idx = [], [], [], []
        for i, smi in enumerate(tqdm(df[smiles_col].tolist(), desc="Featurizing",
                                     unit="mol", leave=True)):
            mol = self.standardize(smi)
            if mol is None:
                continue
            mols.append(mol)
            fps.append(self.morgan_fp(mol))
            desc_rows.append(self.descriptors(mol))
            keep_idx.append(i)
        desc_arr = np.vstack(desc_rows) if desc_rows else np.empty((0, len(self.DESCRIPTOR_NAMES)))
        clean_df = df.iloc[keep_idx].reset_index(drop=True)
        n_dropped = len(df) - len(clean_df)
        if n_dropped:
            print(f"  dropped {n_dropped} molecules that failed standardization/parsing.")
        return {"df": clean_df, "mols": mols, "fps": fps, "descriptors": desc_arr,
                "descriptor_names": self.DESCRIPTOR_NAMES}


featurizer = RDKitFeaturizer(CONFIG)
pretrain_feat = featurizer.featurize_dataframe(pretrain_df, smiles_col="smiles")
train_feat = featurizer.featurize_dataframe(train_df, smiles_col="smiles")
screen_feat = featurizer.featurize_dataframe(screen_df, smiles_col="smiles")

pretrain_df = pretrain_feat["df"]
train_df = train_feat["df"]
screen_df = screen_feat["df"]

pretrain_fps, pretrain_desc_raw = pretrain_feat["fps"], pretrain_feat["descriptors"]
train_fps, train_desc_raw = train_feat["fps"], train_feat["descriptors"]
screen_fps, screen_desc_raw = screen_feat["fps"], screen_feat["descriptors"]
descriptor_names_raw = pretrain_feat["descriptor_names"]

print(f"ChEMBL pretraining set after featurization: {len(pretrain_df):,}")
print(f"BindingDB PSMA fine-tuning set after featurization: {len(train_df):,}")
print(f"Screening set after featurization: {len(screen_df):,}")
print(f"Raw descriptor pool: {len(descriptor_names_raw)} descriptors (before redundancy filtering below).")


In [ ]:
print(CONFIG.screening_library_path)

## 5b. Part B.1b — Descriptor curation (near-constant + correlation-cluster filtering)

Directly follows the AutoQSAR paper's two-stage descriptor curation, applied to the
broad ChEMBL pretraining set (the largest, most chemically diverse sample, so its
correlation structure generalizes best):

1. **Drop near-constant / near-zero-variance descriptors** — a descriptor where ≥90%
   of compounds share one value carries almost no information (e.g., "number of
   tetrazoles" is 0 for nearly everything).
2. **Cluster the survivors by |Pearson correlation|** (hierarchical, complete-linkage,
   cut at r = 0.8) and keep only the descriptor closest to each cluster's centroid.
   This directly addresses the collinearity in a ~200-descriptor pool without
   depending on any one model family to "figure it out" from redundant inputs.

The same descriptor columns (fit once, on the pretraining set) are then applied to
the fine-tuning and screening descriptor matrices, so everything downstream — the
shared `feature_scaler`, the NN trunk, the GBM/KPLS baselines — sees a consistent,
de-redundified feature space.


In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


class AutoQSARDescriptorSelector:
    '''Near-constant filter + correlation-cluster redundancy reduction, as described in
    the AutoQSAR paper (Dixon et al., Future Med. Chem. 2016).'''

    def __init__(self, near_constant_frac: float = 0.90, corr_threshold: float = 0.8):
        self.near_constant_frac = near_constant_frac
        self.corr_threshold = corr_threshold
        self.selected_idx: Optional[np.ndarray] = None
        self.selected_names: Optional[List[str]] = None

    def fit(self, X: np.ndarray, names: Sequence[str]) -> "AutoQSARDescriptorSelector":
        X = np.nan_to_num(X)
        n = len(X)

        # Stage 1: near-constant filter.
        keep1 = []
        for j in range(X.shape[1]):
            col = X[:, j]
            _, counts = np.unique(col, return_counts=True)
            most_common_frac = counts.max() / n
            if most_common_frac < self.near_constant_frac and np.var(col) > 1e-8:
                keep1.append(j)
        if not keep1:
            keep1 = list(range(X.shape[1]))
        X1 = X[:, keep1]
        names1 = [names[j] for j in keep1]
        print(f"  Stage 1 (near-constant filter): {X.shape[1]} -> {X1.shape[1]} descriptors.")

        # Stage 2: correlation-cluster redundancy reduction.
        if X1.shape[1] > 1:
            corr = np.corrcoef(X1, rowvar=False)
            corr = np.nan_to_num(corr, nan=0.0)
            np.fill_diagonal(corr, 1.0)
            dist = 1.0 - np.abs(corr)
            dist = (dist + dist.T) / 2.0        # enforce exact symmetry (float noise)
            np.fill_diagonal(dist, 0.0)
            condensed = squareform(dist, checks=False)
            Z = linkage(condensed, method="complete")
            cluster_labels = fcluster(Z, t=1.0 - self.corr_threshold, criterion="distance")

            keep2 = []
            for cl in np.unique(cluster_labels):
                members = np.where(cluster_labels == cl)[0]
                if len(members) == 1:
                    keep2.append(members[0])
                    continue
                sub_corr = np.abs(corr[np.ix_(members, members)])
                centroid_scores = sub_corr.mean(axis=1)   # closest-to-centroid = highest mean similarity
                keep2.append(members[np.argmax(centroid_scores)])
            keep2 = sorted(keep2)
        else:
            keep2 = list(range(X1.shape[1]))

        self.selected_idx = np.array([keep1[j] for j in keep2])
        self.selected_names = [names[j] for j in self.selected_idx]
        print(f"  Stage 2 (correlation clustering, r<={self.corr_threshold}): "
              f"{X1.shape[1]} -> {len(self.selected_idx)} descriptors.")
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return np.nan_to_num(X)[:, self.selected_idx]


print("Fitting AutoQSAR-style descriptor selector on the broad ChEMBL pretraining set...")
descriptor_selector = AutoQSARDescriptorSelector(corr_threshold=0.8).fit(pretrain_desc_raw, descriptor_names_raw)

pretrain_desc = descriptor_selector.transform(pretrain_desc_raw)
train_desc = descriptor_selector.transform(train_desc_raw)
screen_desc = descriptor_selector.transform(screen_desc_raw)
descriptor_names = descriptor_selector.selected_names

print(f"Final curated descriptor set ({len(descriptor_names)}): {descriptor_names[:12]}"
      f"{' ...' if len(descriptor_names) > 12 else ''}")


## 6. Part B.2 — Applicability Domain

Unchanged logic from the original pipeline: four complementary distance-from-training
metrics (Tanimoto dissimilarity on fingerprints, Manhattan distance in descriptor
space, Manhattan distance in a PCA "latent" proxy, and k-NN unfamiliarity), each
calibrated against a percentile of the **training set's own internal nearest-neighbor
distances**. Calibrated on the **PSMA-specific BindingDB fine-tuning set** — that's
the domain the final predictions need to be trustworthy within, not the broad ChEMBL
pretraining set.


In [ ]:
def fps_to_array(fps) -> np.ndarray:
    arr = np.zeros((len(fps), fps[0].GetNumBits()), dtype=np.uint8)
    for i, fp in enumerate(fps):
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return arr


def bulk_min_tanimoto_dissimilarity(query_fps, ref_fps,
                                     desc: str = "Tanimoto to nearest reference") -> np.ndarray:
    out = np.empty(len(query_fps))
    for i, qfp in enumerate(tqdm(query_fps, desc=desc, unit="mol", leave=False)):
        sims = DataStructs.BulkTanimotoSimilarity(qfp, ref_fps)
        out[i] = 1.0 - max(sims)
    return out


def train_internal_tanimoto_dissimilarity(train_fps) -> np.ndarray:
    out = np.empty(len(train_fps))
    for i, qfp in enumerate(tqdm(train_fps, desc="AD calibration: internal NN Tanimoto",
                                  unit="mol", leave=False)):
        sims = DataStructs.BulkTanimotoSimilarity(qfp, train_fps)
        sims[i] = -1
        out[i] = 1.0 - max(sims)
    return out


class ApplicabilityDomain:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.scaler: Optional[StandardScaler] = None
        self.pca: Optional[PCA] = None
        self.thresholds: dict = {}

    def fit(self, train_fps, train_desc: np.ndarray):
        self.scaler = StandardScaler().fit(np.nan_to_num(train_desc))
        train_desc_scaled = self.scaler.transform(np.nan_to_num(train_desc))

        n_comp = min(self.cfg.latent_pca_components, train_desc_scaled.shape[1], train_desc_scaled.shape[0] - 1)
        self.pca = PCA(n_components=max(n_comp, 1), random_state=self.cfg.random_state).fit(train_desc_scaled)
        train_latent = self.pca.transform(train_desc_scaled)

        tanimoto_internal = train_internal_tanimoto_dissimilarity(train_fps)
        self.thresholds["tanimoto"] = np.percentile(tanimoto_internal, self.cfg.ad_percentile_threshold)

        nn_desc_l1 = NearestNeighbors(n_neighbors=2, p=1).fit(train_desc_scaled)
        d_l1, _ = nn_desc_l1.kneighbors(train_desc_scaled)
        self.thresholds["descriptor_manhattan"] = np.percentile(d_l1[:, 1], self.cfg.ad_percentile_threshold)

        nn_latent_l1 = NearestNeighbors(n_neighbors=2, p=1).fit(train_latent)
        d_lat, _ = nn_latent_l1.kneighbors(train_latent)
        self.thresholds["latent_manhattan"] = np.percentile(d_lat[:, 1], self.cfg.ad_percentile_threshold)

        k = min(self.cfg.unfamiliarity_k + 1, len(train_desc_scaled))
        nn_unfam = NearestNeighbors(n_neighbors=k).fit(train_desc_scaled)
        d_unfam, _ = nn_unfam.kneighbors(train_desc_scaled)
        unfam_internal = d_unfam[:, 1:].mean(axis=1)
        self.thresholds["unfamiliarity"] = np.percentile(unfam_internal, self.cfg.ad_percentile_threshold)

        self._train_desc_scaled = train_desc_scaled
        self._train_latent = train_latent
        self._nn_desc_l1 = NearestNeighbors(n_neighbors=1, p=1).fit(train_desc_scaled)
        self._nn_latent_l1 = NearestNeighbors(n_neighbors=1, p=1).fit(train_latent)
        self._nn_unfam = NearestNeighbors(n_neighbors=min(self.cfg.unfamiliarity_k, len(train_desc_scaled))).fit(
            train_desc_scaled
        )
        return self

    def score(self, screen_fps, screen_desc: np.ndarray, train_fps) -> pd.DataFrame:
        screen_desc_scaled = self.scaler.transform(np.nan_to_num(screen_desc))
        screen_latent = self.pca.transform(screen_desc_scaled)

        tanimoto_diss = bulk_min_tanimoto_dissimilarity(
            screen_fps, train_fps,
            desc="AD: Tanimoto to nearest training compound")
        with tqdm(total=3, desc="AD: descriptor / latent / k-NN distances",
                  leave=False) as _ad_bar:
            desc_manhattan, _ = self._nn_desc_l1.kneighbors(screen_desc_scaled)
            _ad_bar.update(1)
            latent_manhattan, _ = self._nn_latent_l1.kneighbors(screen_latent)
            _ad_bar.update(1)
            unfam_d, _ = self._nn_unfam.kneighbors(screen_desc_scaled)
            _ad_bar.update(1)
        unfamiliarity = unfam_d.mean(axis=1)

        out = pd.DataFrame({
            "tanimoto_dissimilarity": tanimoto_diss,
            "descriptor_manhattan": desc_manhattan[:, 0],
            "latent_manhattan": latent_manhattan[:, 0],
            "unfamiliarity": unfamiliarity,
        })
        out["tanimoto_OOD"] = out["tanimoto_dissimilarity"] > self.thresholds["tanimoto"]
        out["descriptor_OOD"] = out["descriptor_manhattan"] > self.thresholds["descriptor_manhattan"]
        out["latent_OOD"] = out["latent_manhattan"] > self.thresholds["latent_manhattan"]
        out["unfamiliarity_OOD"] = out["unfamiliarity"] > self.thresholds["unfamiliarity"]
        out["n_flags_OOD"] = out[["tanimoto_OOD", "descriptor_OOD", "latent_OOD", "unfamiliarity_OOD"]].sum(axis=1)

        # F6: Tanimoto is a VETO, not one vote in four.
        #
        # F5 used `n_flags_OOD >= 3`, which let a compound pass while failing the
        # structural-similarity test - and for a congeneric target that is the flag with
        # teeth. The other three metrics compare bulk physicochemical properties, so a
        # molecule of roughly the right size and lipophilicity passes them while being
        # structurally unrelated to anything in training. Every one of F5's 15 top hits
        # was flagged tanimoto_OOD (dissimilarity 0.73-0.83 against a 0.457 threshold)
        # and every one of them was reported as in-domain. Meanwhile the Butina-grouped
        # evaluation showed that this is exactly the regime where the model fails.
        out["consensus_OOD"] = out["n_flags_OOD"] >= self.cfg.ad_consensus_min_flags
        if getattr(self.cfg, "ad_tanimoto_veto", False):
            out["consensus_OOD"] = out["consensus_OOD"] | out["tanimoto_OOD"]
        return out


ad = ApplicabilityDomain(CONFIG).fit(train_fps, train_desc)
ad_scores = ad.score(screen_fps, screen_desc, train_fps)
print("AD thresholds (95th percentile of the PSMA training set's own internal NN distances):")
for k, v in ad.thresholds.items():
    print(f"  {k}: {v:.4f}")

_n = len(ad_scores)
print(f"\nScreening library: {_n:,} compounds")
for flag in ("tanimoto_OOD", "descriptor_OOD", "latent_OOD", "unfamiliarity_OOD"):
    print(f"  {flag:>18s} fires on {ad_scores[flag].sum():>7,} "
          f"({100 * ad_scores[flag].mean():5.1f}%)")
_veto = getattr(CONFIG, "ad_tanimoto_veto", False)
_in_dom = int((~ad_scores["consensus_OOD"]).sum())
print(f"\n  Tanimoto veto: {'ON' if _veto else 'OFF'}"
      f"   (consensus_min_flags = {CONFIG.ad_consensus_min_flags})")
print(f"  -> in-domain: {_in_dom:,} / {_n:,} ({100 * _in_dom / _n:.2f}%)")
_would_be = int((ad_scores["n_flags_OOD"] < CONFIG.ad_consensus_min_flags).sum())
if _veto:
    print(f"     without the veto it would have been {_would_be:,} "
          f"({100 * _would_be / _n:.2f}%) - the veto removes "
          f"{_would_be - _in_dom:,} compounds that passed on bulk-property grounds")
    print("     while being structurally unlike anything in the training set.")
ad_scores.describe()


## 6b. Part B.2b — Which feature set actually works best here?

Both papers found that the best choice of input representation (fingerprints only,
descriptors only, or both together) is target-dependent rather than universal — the
KPLS paper found fingerprints alone beat descriptors alone on 9/10 targets, but the
*combination* won outright for one of them. Rather than assume `CONFIG.feature_set`
is right, a fast default-hyperparameter LightGBM is cross-validated on the PSMA
training data under all three options (scored on `pIC50`, the primary endpoint), and
whichever wins becomes the feature set used for every model built from here on.


In [ ]:
# --- Which representation works best for CLASSIFICATION? -----------------------------
# F5 scored this with LightGBM regression R2. The target is now a class, so the probe is
# a classifier scored by ROC-AUC under stratified CV. Note this uses the full labelled
# set, so it is a (small) selection-on-all-data step: it chooses among three fixed
# options and does not see the eventual held-out folds' labels in any fitted parameter,
# but it is not a clean nested choice either. Left in place for continuity with F5.
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier


def build_design_matrix(fps, desc: np.ndarray, scaler: StandardScaler, feature_set: str) -> np.ndarray:
    desc_scaled = scaler.transform(np.nan_to_num(desc))
    if feature_set == "descriptors_only":
        return desc_scaled
    fp_arr = fps_to_array(fps)
    if feature_set == "fp_only":
        return fp_arr
    if feature_set == "fp_plus_descriptors":
        return np.hstack([fp_arr, desc_scaled])
    raise ValueError(f"Unknown feature_set '{feature_set}'.")


_feature_set_options = ["fp_only", "descriptors_only", "fp_plus_descriptors"]
_quick_scaler = StandardScaler().fit(np.nan_to_num(train_desc))
_y_probe = train_df["y"].values.astype(int)
_skf = StratifiedKFold(n_splits=min(CONFIG.cv_folds, int(np.bincount(_y_probe).min())),
                       shuffle=True, random_state=CONFIG.random_state)

_fs_scores = {}
for fs in tqdm(_feature_set_options, desc="Feature-set bake-off", unit="option"):
    X_probe = build_design_matrix(train_fps, train_desc, _quick_scaler, fs)
    aucs = []
    for tr_idx, te_idx in tqdm(_skf.split(X_probe, _y_probe), total=_skf.get_n_splits(),
                                desc=f"  {fs}: CV folds", leave=False):
        m = LGBMClassifier(random_state=CONFIG.random_state, verbose=-1,
                            n_estimators=150, n_jobs=-1,
                            class_weight="balanced" if CONFIG.class_weight_balanced else None)
        m.fit(X_probe[tr_idx], _y_probe[tr_idx])
        p = m.predict_proba(X_probe[te_idx])
        if p.shape[1] == 2:
            aucs.append(roc_auc_score(_y_probe[te_idx], p[:, 1]))
        else:
            aucs.append(roc_auc_score(_y_probe[te_idx], p, multi_class="ovr",
                                       average="macro"))
    _fs_scores[fs] = float(np.mean(aucs))
    print(f"  {fs:>22s}: mean CV ROC-AUC = {_fs_scores[fs]:.3f}")

_best_feature_set = max(_fs_scores, key=_fs_scores.get)
print(f"\nBest feature set for {PRIMARY} classification: '{_best_feature_set}' "
      f"(CONFIG default was '{CONFIG.feature_set}')")
CONFIG.feature_set = _best_feature_set

## 7. Part B.3 — Design matrices

One shared feature scaler (`feature_scaler`, fit on the **broad ChEMBL pretraining
set**, since that's the larger and more chemically diverse sample) is reused to build
the model-input matrix for all three sets. Using the same scaler everywhere is what
lets the pretrained trunk's weights be meaningfully reused during fine-tuning — the
fine-tuning and screening inputs live in the exact same normalized feature space the
trunk was pretrained on. This is separate from `ad.scaler` above, which is
intentionally calibrated only on the PSMA set for domain-membership scoring.
`CONFIG.feature_set` here is whatever Part B.2b just selected.


In [ ]:
# --- Design matrices and the reference split ------------------------------------------
# One shared scaler fit on the broad ChEMBL pretraining set, reused everywhere, so the
# pretrained trunk's weights stay meaningful once loaded into the fine-tuning model.
feature_scaler = StandardScaler().fit(np.nan_to_num(pretrain_desc))

X_pretrain = build_design_matrix(pretrain_fps, pretrain_desc, feature_scaler, CONFIG.feature_set)
y_pretrain = pretrain_df[CONFIG.chembl_pretrain_endpoint_name].values.astype(np.float32)

X_train_full = build_design_matrix(train_fps, train_desc, feature_scaler, CONFIG.feature_set)
y_train_full = train_df["y"].values.astype(np.int64)

X_screen = build_design_matrix(screen_fps, screen_desc, feature_scaler, CONFIG.feature_set)

# Optional per-compound weights, to blunt a single over-represented source.
def source_weight_vector(df: pd.DataFrame, enabled: bool) -> np.ndarray:
    """Inverse-sqrt-frequency weight per source tag, mean-normalised to 1.

    Exploratory. One patent supplies ~26% of the labels here; whether to discount it is a
    modelling judgement, not a fact, so this is OFF by default and its effect is
    measurable rather than assumed."""
    if not enabled:
        return np.ones(len(df), dtype=np.float32)
    counts = df["source_tag"].map(df["source_tag"].value_counts()).values.astype(np.float64)
    w = 1.0 / np.sqrt(counts)
    w = w / w.mean()
    return w.astype(np.float32)


w_train_full = source_weight_vector(train_df, CONFIG.downweight_dominant_source)

# A single STRATIFIED random split, kept only as a reference point comparable with F5.
# The evaluation that matters is leave-one-cluster-out in Part E.2.
X_tr, X_te, y_tr, y_te, w_tr, w_te, idx_tr, idx_te = train_test_split(
    X_train_full, y_train_full, w_train_full, np.arange(len(y_train_full)),
    test_size=CONFIG.test_size, random_state=CONFIG.random_state,
    stratify=y_train_full,
)

print(f"Feature set: '{CONFIG.feature_set}' -> {X_train_full.shape[1]} input dimensions")
print(f"ChEMBL pretraining: {X_pretrain.shape}")
print(f"PSMA classification — train: {X_tr.shape}, held-out reference test: {X_te.shape}")
for c, name in enumerate(CLASS_NAMES):
    print(f"  {name:>11s}: {int((y_tr == c).sum()):4d} in train, "
          f"{int((y_te == c).sum()):4d} in test")
if CONFIG.downweight_dominant_source:
    print(f"Source down-weighting ON: weights range "
          f"{w_train_full.min():.3f}-{w_train_full.max():.3f}")

## 8. Part C.1 — Transfer-learning classifier

A shared MLP **trunk** with two heads: a single-output regression head used only while
pretraining on the broad ChEMBL potency set, and a classification head used during
fine-tuning on the PSMA activity classes. Only the trunk transfers, which is the actual
"transfer learning" step — fine-tuning loads its weights, freezes it for
`freeze_epochs` while the new head adapts, then unfreezes for gradual joint fine-tuning.

Two changes from F3–F5 beyond the obvious one. The masked multi-task MSE is gone,
replaced by **class-weighted cross-entropy** on a single endpoint: Part A.2b showed the
second task was largely a Cheng–Prusoff re-expression of the first, and the masked loss
was giving dual-labelled compounds twice the weight of single-labelled ones. And
`predict_proba_mc_dropout` returns a probability spread rather than a value spread —
useful as a signal about model instability, but the probability itself is the honest
uncertainty, and Part E.2d is what tests it.

In [ ]:
# --- Transfer-learning classifier ------------------------------------------------------
class TransferClassifier(nn.Module):
    """Shared MLP trunk + two heads.

    `pretrain_head` is a single-output regression head used only while pretraining on the
    broad ChEMBL potency set (unchanged from F3-F5 — the pretraining labels are real
    numbers, so regression is the right pretext task). `clf_head` is the classification
    head used during fine-tuning on the PSMA activity classes. Only the trunk transfers.
    """

    def __init__(self, n_in: int, hidden_dims=(256, 128), dropout: float = 0.2,
                 n_classes: int = 2):
        super().__init__()
        layers = []
        d = n_in
        for h in hidden_dims:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.trunk = nn.Sequential(*layers)
        self.trunk_out_dim = d
        self.pretrain_head = nn.Linear(d, 1)
        self.clf_head = nn.Linear(d, n_classes)

    def forward_pretrain(self, x):
        return self.pretrain_head(self.trunk(x)).squeeze(-1)

    def forward_logits(self, x):
        return self.clf_head(self.trunk(x))


def pretrain_trunk(X, y, hidden_dims, dropout, lr, weight_decay, epochs, batch_size, seed=0,
                   device=DEVICE, desc="Pretraining trunk on broad ChEMBL data",
                   leave=True) -> TransferClassifier:
    """Unchanged in substance from F5: plain MSE regression on the ChEMBL potency label."""
    torch.manual_seed(seed)
    model = TransferClassifier(X.shape[1], hidden_dims=hidden_dims, dropout=dropout,
                               n_classes=CONFIG.n_activity_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    yt = torch.tensor(y, dtype=torch.float32, device=device)
    n = len(X)
    for _ in tqdm(range(epochs), desc=desc, unit="epoch", leave=leave):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = nn.functional.mse_loss(model.forward_pretrain(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
    return model


def class_weights_for(y: np.ndarray, n_classes: int, enabled: bool) -> Optional[torch.Tensor]:
    if not enabled:
        return None
    counts = np.bincount(y, minlength=n_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    w = len(y) / (n_classes * counts)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def finetune_classifier(pretrained_trunk_state, X, y, hidden_dims, dropout, lr, weight_decay,
                        freeze_epochs, total_epochs, batch_size, sample_weight=None,
                        seed=0, device=DEVICE, desc=None, leave=False) -> TransferClassifier:
    """Loads a pretrained trunk (None = random init, which is the ablation arm), freezes
    it for `freeze_epochs` while the new classification head adapts, then unfreezes for
    gradual fine-tuning. Class-weighted cross-entropy, optionally multiplied by a
    per-compound source weight."""
    torch.manual_seed(seed)
    n_classes = CONFIG.n_activity_classes
    model = TransferClassifier(X.shape[1], hidden_dims=hidden_dims, dropout=dropout,
                               n_classes=n_classes)
    if pretrained_trunk_state is not None:
        model.trunk.load_state_dict(pretrained_trunk_state)
    model = model.to(device)

    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    yt = torch.tensor(y, dtype=torch.long, device=device)
    if sample_weight is None:
        sample_weight = np.ones(len(y), dtype=np.float32)
    swt = torch.tensor(sample_weight, dtype=torch.float32, device=device)
    cw = class_weights_for(y, n_classes, CONFIG.class_weight_balanced)
    n = len(X)

    if desc is None:
        desc = ("Fine-tuning classifier (transfer)" if pretrained_trunk_state is not None
                else "Fine-tuning classifier (random init)")
    loss = None
    _bar = tqdm(range(total_epochs), desc=desc, unit="epoch", leave=leave)
    for ep in _bar:
        frozen = ep < freeze_epochs
        for p in model.trunk.parameters():
            p.requires_grad = not frozen
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            logits = model.forward_logits(Xt[idx])
            per_sample = nn.functional.cross_entropy(logits, yt[idx], weight=cw,
                                                     reduction="none")
            loss = (per_sample * swt[idx]).sum() / swt[idx].sum().clamp(min=1e-8)
            loss.backward()
            opt.step()
        _bar.set_postfix(loss=("n/a" if loss is None else f"{loss.item():.3f}"),
                         trunk="frozen" if frozen else "unfrozen", refresh=False)
    _bar.close()
    return model


def predict_proba(model: TransferClassifier, X, device=DEVICE) -> np.ndarray:
    """Deterministic class probabilities (dropout off)."""
    model = model.to(device).eval()
    with torch.no_grad():
        logits = model.forward_logits(torch.tensor(X, dtype=torch.float32, device=device))
        return torch.softmax(logits, dim=1).cpu().numpy()


def predict_proba_mc_dropout(model: TransferClassifier, X, n_passes: int,
                             device=DEVICE) -> Tuple[np.ndarray, np.ndarray]:
    """(mean, std) class probabilities over `n_passes` stochastic passes with dropout on.

    A caution carried over from F5: Optuna tuned dropout to ~0.047, which leaves this
    almost no variance to report. The probability itself is the honest uncertainty here;
    the MC spread is a secondary signal about model instability, not about how wrong the
    prediction is. Part E.3's reliability diagram is the test that matters."""
    model = model.to(device)
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    model.train()
    with torch.no_grad():
        probs = torch.stack([
            torch.softmax(model.forward_logits(Xt), dim=1)
            for _ in tqdm(range(n_passes), desc="MC-dropout passes", unit="pass",
                           leave=False)
        ], dim=0).cpu().numpy()
    model.eval()
    return probs.mean(axis=0), probs.std(axis=0)


print(f"Transfer classifier defined: TransferClassifier ({CONFIG.n_activity_classes} "
      f"classes), pretrain_trunk, finetune_classifier, predict_proba, "
      f"predict_proba_mc_dropout. Device: {DEVICE}.")

## 9. Part C.2 — Baselines: LightGBM and kernel PLS-DA

Two non-transfer comparators fitted directly on the PSMA classification set.

- **LightGBM** — `LGBMClassifier`, class-weighted.
- **Kernel PLS-DA** — following An, Sherman & Dixon (*J. Chem. Inf. Model.* 2013): a
  Nystroem low-rank approximation of a Gaussian kernel, then PLS to a small latent space.
  F5 fed those components to `PLSRegression` against a continuous target; for a class
  label the analogue is PLS-DA, so the PLS scores now feed a multinomial logistic layer,
  which keeps the same dimensionality reduction while producing probabilities.

Both get stratified bootstrap ensembles, so a resample cannot lose a class. Each model
exposes a `*_predict_proba` wrapper that pads any class a training fold never saw, so
every committee member returns the same shape and the equal-weight blend is always
well defined — leave-one-cluster-out will produce folds that are badly unbalanced.

In [ ]:
# --- Baselines: LightGBM and kernel PLS-DA, both as classifiers -----------------------
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def fit_lgbm_classifier(X, y, params: dict, sample_weight=None, seed: int = 42):
    model = LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1,
                           class_weight="balanced" if CONFIG.class_weight_balanced else None,
                           **params)
    model.fit(X, y, sample_weight=sample_weight)
    return model


def make_kpls_classifier(n_components: int, gamma: float, seed: int = 42) -> Pipeline:
    """Kernel PLS-DA: the Nystroem low-rank kernel map of An/Sherman/Dixon 2013, then PLS
    to a small latent space, then multinomial logistic regression on those components.

    F5 used Nystroem -> PLSRegression on a continuous target. For a class label the
    natural analogue is PLS-DA; routing the PLS scores into a logistic layer keeps the
    same dimensionality reduction while producing calibrated-ish probabilities."""
    n_comp = max(1, min(n_components, 50))
    return Pipeline([
        ("nystroem", Nystroem(kernel="rbf", gamma=gamma, n_components=200,
                              random_state=seed)),
        ("pls", PLSRegression(n_components=n_comp)),
        ("logit", LogisticRegression(max_iter=2000,
                                     class_weight="balanced" if CONFIG.class_weight_balanced
                                     else None)),
    ])


def fit_kpls_classifier(X, y, params: dict, sample_weight=None, seed: int = 42) -> Pipeline:
    # PLSRegression is a transformer here, so the pipeline is fitted in two stages: PLS
    # needs the target to project against, the logistic layer needs the projection.
    # Nystroem cannot draw more landmarks than it has samples. Leave-one-cluster-out
    # produces small training halves, so clamp rather than warn-and-silently-degrade.
    n_landmarks = int(min(200, max(2, len(X) - 1)))
    nys = Nystroem(kernel="rbf", gamma=params["gamma"], n_components=n_landmarks,
                   random_state=seed)
    Z = nys.fit_transform(X)
    n_comp = max(1, min(params["n_components"], min(50, Z.shape[1] - 1)))
    pls = PLSRegression(n_components=n_comp)
    # one-hot the target so PLS has something multivariate to align with
    Y1h = np.zeros((len(y), CONFIG.n_activity_classes), dtype=float)
    Y1h[np.arange(len(y)), y] = 1.0
    pls.fit(Z, Y1h)
    T = pls.transform(Z)
    logit = LogisticRegression(max_iter=2000,
                               class_weight="balanced" if CONFIG.class_weight_balanced
                               else None)
    logit.fit(T, y, sample_weight=sample_weight)
    return {"nystroem": nys, "pls": pls, "logit": logit, "classes_": logit.classes_}


def kpls_predict_proba(model, X: np.ndarray) -> np.ndarray:
    """Returns an (n, n_classes) probability matrix, padding any class the training fold
    never saw so every model in the committee returns the same shape."""
    T = model["pls"].transform(model["nystroem"].transform(X))
    p = model["logit"].predict_proba(T)
    return _align_proba(p, model["classes_"])


def _align_proba(p: np.ndarray, classes) -> np.ndarray:
    """Expand a classifier's proba matrix to the full class set (missing classes -> 0)."""
    n_classes = CONFIG.n_activity_classes
    if p.shape[1] == n_classes and list(classes) == list(range(n_classes)):
        return p
    out = np.zeros((p.shape[0], n_classes))
    for j, c in enumerate(classes):
        out[:, int(c)] = p[:, j]
    return out


def lgbm_predict_proba(model, X: np.ndarray) -> np.ndarray:
    return _align_proba(model.predict_proba(X), model.classes_)


def bootstrap_lgbm_classifiers(X, y, params: dict, n_bootstrap: int, sample_weight=None,
                               seed: int = 42) -> List:
    """Stratified bootstrap, so a resample cannot lose a class entirely."""
    rng = np.random.RandomState(seed)
    by_class = {c: np.flatnonzero(y == c) for c in np.unique(y)}
    models = []
    for b in tqdm(range(n_bootstrap), desc="Bootstrap LightGBM ensemble", unit="model",
                   leave=False):
        idx = np.concatenate([rng.choice(v, size=len(v), replace=True)
                              for v in by_class.values()])
        sw = None if sample_weight is None else sample_weight[idx]
        models.append(fit_lgbm_classifier(X[idx], y[idx], params, sample_weight=sw,
                                          seed=seed + b))
    return models


def bootstrap_kpls_classifiers(X, y, params: dict, n_bootstrap: int, sample_weight=None,
                               seed: int = 42) -> List:
    rng = np.random.RandomState(seed)
    by_class = {c: np.flatnonzero(y == c) for c in np.unique(y)}
    models = []
    for b in tqdm(range(n_bootstrap), desc="Bootstrap KPLS-DA ensemble", unit="model",
                   leave=False):
        idx = np.concatenate([rng.choice(v, size=len(v), replace=True)
                              for v in by_class.values()])
        sw = None if sample_weight is None else sample_weight[idx]
        try:
            models.append(fit_kpls_classifier(X[idx], y[idx], params, sample_weight=sw,
                                              seed=seed + b))
        except Exception:
            continue
    return models


print("Baseline helpers defined: fit_lgbm_classifier, fit_kpls_classifier, "
      "lgbm_predict_proba, kpls_predict_proba, bootstrap_* ensembles.")

## 9b. Part C.2b — Classification and screening metrics

Defined before any model is fitted so the evaluation protocol is fixed in advance.

`classification_metrics` returns ROC-AUC, PR-AUC, balanced accuracy, MCC, Brier score,
precision, recall and F1 for one (fold, model) pair, and returns NaN rather than raising
on a single-class fold — which leave-one-cluster-out will certainly produce.

Two of these are new to the project and matter most for its actual purpose.
**`enrichment_factor`** is the hit rate in the top fraction of a ranking divided by the
overall hit rate: EF = 1 means the ranking is worthless, EF = 10 at 1% means the top
percentile is ten times richer in actives than the library. **`precision_at_n`** is the
blunter version — of the top n ranked compounds, what fraction are genuinely active. No
earlier version of this pipeline computed anything like either, which meant there was no
number to answer the only question a screen is for.

`blend_proba_equal` is the committee combiner. F5 fitted simplex blend weights; a plain
average beat them in all six scheme × endpoint comparisons, by up to 0.08 R² under
grouped splits where the inner validation slice is smallest. The optimizer is gone rather
than kept as an option — it had no winning case, and it was also the leakage surface that
let F5's Part E.2 fit weights on the very test set it then scored against.

In [ ]:
# --- Classification and screening metrics ----------------------------------------------
from sklearn.metrics import (roc_auc_score, average_precision_score, balanced_accuracy_score,
                              matthews_corrcoef, confusion_matrix, brier_score_loss,
                              precision_score, recall_score, f1_score)


def _auc_any(y_true, proba) -> float:
    """ROC-AUC that works for 2 or 3 classes and returns NaN rather than raising when a
    fold is single-class (which leave-one-cluster-out will produce)."""
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    proba = np.asarray(proba)
    if CONFIG.n_activity_classes == 2:
        return float(roc_auc_score(y_true, proba[:, POSITIVE_CLASS]))
    try:
        present = np.unique(y_true)
        p = proba[:, present] / proba[:, present].sum(axis=1, keepdims=True)
        return float(roc_auc_score(y_true, p, multi_class="ovr", average="macro",
                                   labels=present))
    except Exception:
        return float("nan")


def enrichment_factor(y_true, scores, frac: float) -> float:
    """EF@frac = (hit rate in the top `frac` of the ranking) / (overall hit rate).

    The screening metric the earlier pipelines never computed. EF = 1 means the ranking is
    worthless; EF = 10 at 1% means the top percentile is ten times richer in actives than
    the library as a whole."""
    y_true = np.asarray(y_true)
    base = (y_true == POSITIVE_CLASS).mean()
    if base == 0 or len(y_true) == 0:
        return float("nan")
    k = max(1, int(round(frac * len(y_true))))
    top = np.argsort(-np.asarray(scores))[:k]
    return float((y_true[top] == POSITIVE_CLASS).mean() / base)


def precision_at_n(y_true, scores, n: int) -> float:
    """Of the top n ranked compounds, what fraction are actually active."""
    y_true = np.asarray(y_true)
    n = min(n, len(y_true))
    if n == 0:
        return float("nan")
    top = np.argsort(-np.asarray(scores))[:n]
    return float((y_true[top] == POSITIVE_CLASS).mean())


def classification_metrics(y_true, proba, decision_threshold: float = 0.5) -> Dict[str, float]:
    """Everything worth reporting for one (fold, model) pair."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    p_pos = proba[:, POSITIVE_CLASS]
    if CONFIG.n_activity_classes == 2:
        y_pred = (p_pos >= decision_threshold).astype(int)
    else:
        y_pred = proba.argmax(axis=1)

    out = {
        "n": int(len(y_true)),
        "n_active": int((y_true == POSITIVE_CLASS).sum()),
        "base_rate": float((y_true == POSITIVE_CLASS).mean()),
        "roc_auc": _auc_any(y_true, proba),
        "balanced_acc": (float(balanced_accuracy_score(y_true, y_pred))
                         if len(np.unique(y_true)) > 1 else float("nan")),
        "mcc": (float(matthews_corrcoef(y_true, y_pred))
                if len(np.unique(y_true)) > 1 else float("nan")),
    }
    if CONFIG.n_activity_classes == 2:
        y_bin = (y_true == POSITIVE_CLASS).astype(int)
        out["pr_auc"] = (float(average_precision_score(y_bin, p_pos))
                         if y_bin.sum() not in (0, len(y_bin)) else float("nan"))
        out["brier"] = float(brier_score_loss(y_bin, np.clip(p_pos, 0, 1)))
        out["precision"] = float(precision_score(y_bin, y_pred, zero_division=0))
        out["recall"] = float(recall_score(y_bin, y_pred, zero_division=0))
        out["f1"] = float(f1_score(y_bin, y_pred, zero_division=0))
    for frac in CONFIG.enrichment_fractions:
        out[f"EF@{int(frac * 100)}%"] = enrichment_factor(y_true, p_pos, frac)
    return out


def blend_proba_equal(proba_list: List[np.ndarray]) -> np.ndarray:
    """Equal-weight probability average.

    F5 fitted simplex blend weights and a plain average beat them in all six
    scheme x endpoint comparisons - by up to 0.08 R2 under grouped splits, where the inner
    validation slice is smallest and least representative. The weight optimizer has
    therefore been removed rather than kept as an option: it had no winning case, and it
    was also a leakage surface, since F5's Part E.2 fitted it on the very test set it then
    reported scores against."""
    stacked = np.stack([np.asarray(p) for p in proba_list], axis=0)
    out = stacked.mean(axis=0)
    return out / out.sum(axis=1, keepdims=True)


print("Metrics defined: classification_metrics, enrichment_factor, precision_at_n, "
      "blend_proba_equal, _auc_any.")
print(f"Reporting EF at {', '.join(f'{int(f * 100)}%' for f in CONFIG.enrichment_fractions)}"
      f" and precision@{CONFIG.precision_at_n}.")

## 10. Part D.1 — Hyperparameter tuning: LightGBM and KPLS-DA

Same two-search structure as F5 — exhaustive `GridSearchCV` against Bayesian
`BayesSearchCV` — with stratified folds and ROC-AUC in place of R², ranked by the
AutoQSAR train/test-gap-aware score.

One caveat carried forward from the F4 review: that gap penalty systematically prefers
under-fit models. On F4's pKi data it picked a 50-tree, 7-leaf LightGBM over a config
with a higher test score, purely because its train/test gap was narrower. Both candidates
and both raw scores are printed, and the cell says so explicitly when the AutoQSAR winner
is not the model with the best held-out AUC.

In [ ]:
# --- Hyperparameter tuning, scored for classification ---------------------------------
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from skopt import BayesSearchCV
from skopt.space import Real, Integer


def autoqsar_score(accuracy_train: float, accuracy_test: float) -> float:
    """AutoQSAR's model-quality score (Dixon et al. 2016, Eq. 1): rewards models accurate
    on both train and test, penalised by the train/test gap. Unchanged from F5 except
    that "accuracy" is now ROC-AUC rather than R2, and the degenerate floor is 0.5
    (coin-flip) rather than 0.

    A caveat carried forward from the F4 review: this gap penalty systematically prefers
    under-fit models. On F4's pKi data it chose a 50-tree/7-leaf LightGBM over a
    better-generalising config purely because its gap was smaller. Both candidates are
    printed below so the choice stays visible."""
    if accuracy_test < 0.5:
        return accuracy_test - 0.5
    return (accuracy_test - 0.5) * (1.0 - abs(accuracy_train - accuracy_test)) + 0.5


def pick_best_by_autoqsar_score(cv_results: dict):
    train_scores = cv_results["mean_train_score"]
    test_scores = cv_results["mean_test_score"]
    aq = [autoqsar_score(tr, te) for tr, te in zip(train_scores, test_scores)]
    best_idx = int(np.argmax(aq))
    return best_idx, aq[best_idx], float(train_scores[best_idx]), float(test_scores[best_idx])


GRID_PARAMS = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 6, 10],
}
BAYES_SPACE = {
    "n_estimators": Integer(50, 500),
    "learning_rate": Real(0.005, 0.3, prior="log-uniform"),
    "num_leaves": Integer(7, 63),
    "max_depth": Integer(3, 12),
    "min_child_samples": Integer(5, 40),
}
KPLS_GRID_PARAMS = {
    "n_components": [2, 4, 6, 8, 12],
    "gamma": [1.0 / X_tr.shape[1], 0.01, 0.05, 0.1],
}

_SCORING = "roc_auc" if CONFIG.n_activity_classes == 2 else "roc_auc_ovr_weighted"
_n_splits = min(CONFIG.cv_folds, int(np.bincount(y_tr).min()))
_skf_tune = StratifiedKFold(n_splits=_n_splits, shuffle=True,
                            random_state=CONFIG.random_state)
print(f"Tuning with {_n_splits}-fold stratified CV, scoring = '{_SCORING}' "
      f"({int((y_tr == POSITIVE_CLASS).sum())} actives of {len(y_tr)} training rows)\n")

candidates = {}
_search_bar = tqdm(total=int(CONFIG.run_gridsearch) + int(CONFIG.run_bayes_search) + 1,
                   desc="Parameter searches", unit="search")

if CONFIG.run_gridsearch:
    gcv = GridSearchCV(
        LGBMClassifier(random_state=CONFIG.random_state, verbose=-1, n_jobs=-1,
                       class_weight="balanced" if CONFIG.class_weight_balanced else None),
        GRID_PARAMS, cv=_skf_tune, scoring=_SCORING, n_jobs=-1, return_train_score=True)
    gcv.fit(X_tr, y_tr)
    idx, aq, tr_s, te_s = pick_best_by_autoqsar_score(gcv.cv_results_)
    candidates["GridSearchCV"] = (aq, gcv.cv_results_["params"][idx], tr_s, te_s)
    _search_bar.update(1)
    tqdm.write(f"  LightGBM GridSearchCV  AutoQSAR={aq:.3f} (train AUC={tr_s:.3f}, "
               f"test AUC={te_s:.3f})  params={gcv.cv_results_['params'][idx]}")

if CONFIG.run_bayes_search:
    bcv = BayesSearchCV(
        LGBMClassifier(random_state=CONFIG.random_state, verbose=-1, n_jobs=-1,
                       class_weight="balanced" if CONFIG.class_weight_balanced else None),
        BAYES_SPACE, n_iter=CONFIG.bayes_search_n_iter, cv=_skf_tune, scoring=_SCORING,
        n_jobs=-1, random_state=CONFIG.random_state, return_train_score=True)
    bcv.fit(X_tr, y_tr)
    idx, aq, tr_s, te_s = pick_best_by_autoqsar_score(bcv.cv_results_)
    candidates["BayesSearchCV"] = (aq, dict(bcv.cv_results_["params"][idx]), tr_s, te_s)
    _search_bar.update(1)
    tqdm.write(f"  LightGBM BayesSearchCV AutoQSAR={aq:.3f} (train AUC={tr_s:.3f}, "
               f"test AUC={te_s:.3f})  params={dict(bcv.cv_results_['params'][idx])}")

_winner = max(candidates, key=lambda k: candidates[k][0])
gbm_best_params = candidates[_winner][1]
gbm_tuning_results = candidates
tqdm.write(f"  -> LightGBM winner: {_winner} (AutoQSAR={candidates[_winner][0]:.3f})")
_by_auc = max(candidates, key=lambda k: candidates[k][3])
if _by_auc != _winner:
    tqdm.write(f"     NOTE: {_by_auc} had the higher raw test AUC "
               f"({candidates[_by_auc][3]:.3f} vs {candidates[_winner][3]:.3f}) but a "
               f"wider train/test gap. The AutoQSAR penalty chose the narrower-gap model.")

# --- KPLS-DA: small space, grid-searched by hand (it is not a plain sklearn estimator) --
_kpls_rows = []
for nc in KPLS_GRID_PARAMS["n_components"]:
    for gm in KPLS_GRID_PARAMS["gamma"]:
        tr_aucs, te_aucs = [], []
        for tr_i, te_i in _skf_tune.split(X_tr, y_tr):
            try:
                m = fit_kpls_classifier(X_tr[tr_i], y_tr[tr_i],
                                        {"n_components": nc, "gamma": gm},
                                        seed=CONFIG.random_state)
                p_tr = kpls_predict_proba(m, X_tr[tr_i])
                p_te = kpls_predict_proba(m, X_tr[te_i])
            except Exception:
                continue
            tr_aucs.append(_auc_any(y_tr[tr_i], p_tr))
            te_aucs.append(_auc_any(y_tr[te_i], p_te))
        if not te_aucs:
            continue
        tr_m, te_m = float(np.nanmean(tr_aucs)), float(np.nanmean(te_aucs))
        _kpls_rows.append({"n_components": nc, "gamma": gm, "train_auc": tr_m,
                           "test_auc": te_m, "autoqsar": autoqsar_score(tr_m, te_m)})
_search_bar.update(1)
_search_bar.close()

kpls_tuning_results = pd.DataFrame(_kpls_rows).sort_values("autoqsar", ascending=False)
if len(kpls_tuning_results):
    _b = kpls_tuning_results.iloc[0]
    kpls_best_params = {"n_components": int(_b["n_components"]), "gamma": float(_b["gamma"])}
    print(f"  KPLS-DA grid           AutoQSAR={_b['autoqsar']:.3f} "
          f"(train AUC={_b['train_auc']:.3f}, test AUC={_b['test_auc']:.3f})  "
          f"params={kpls_best_params}")
else:
    kpls_best_params = {"n_components": 4, "gamma": 1.0 / X_tr.shape[1]}
    print(f"  KPLS-DA grid failed everywhere; falling back to {kpls_best_params}")

print("\nFinal tuned params:")
print("  LightGBM:", gbm_best_params)
print("  KPLS-DA :", kpls_best_params)

## 11. Part D.2 — Hyperparameter tuning: the classifier head (Optuna, TPE)

`GridSearchCV` does not apply cleanly to a pretrain-then-fine-tune network, so its
architecture and fine-tuning hyperparameters go to Optuna's Tree-structured Parzen
Estimator, scored by held-out ROC-AUC on a stratified validation slice of the training
split.

Watch the dropout it picks. In F5 Optuna chose 0.047, which left MC-dropout with
essentially no variance to report and was a large part of why the uncertainty estimates
came out roughly four times too tight. The cell warns when the chosen dropout falls below
0.1 — that is not a bug to fix by overriding the search, but a reason to trust the
reliability diagram rather than the MC spread.

In [ ]:
# --- Optuna TPE search for the fine-tuning hyperparameters -----------------------------
# Scored by held-out ROC-AUC on a stratified validation slice of the training split.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(
    X_tr, y_tr, test_size=1.0 / CONFIG.optuna_cv_folds,
    random_state=CONFIG.random_state, stratify=y_tr,
)

_default_hidden = tuple(CONFIG.nn_hidden_dims)
_quick_pretrain_epochs = max(5, CONFIG.nn_pretrain_epochs // 4)
_base_pretrained = pretrain_trunk(
    X_pretrain, y_pretrain, hidden_dims=_default_hidden, dropout=CONFIG.nn_dropout,
    lr=CONFIG.nn_lr, weight_decay=CONFIG.nn_weight_decay, epochs=_quick_pretrain_epochs,
    batch_size=CONFIG.nn_pretrain_batch_size, seed=CONFIG.random_state,
    desc="Optuna: base pretrain pass",
)


def optuna_objective(trial: "optuna.Trial") -> float:
    h1 = trial.suggest_categorical("h1", [64, 128, 256])
    h2 = trial.suggest_categorical("h2", [32, 64, 128])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    freeze_epochs = trial.suggest_int("freeze_epochs", 0, 30)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    hidden_dims = (h1, h2)
    trunk_state = (_base_pretrained.trunk.state_dict() if hidden_dims == _default_hidden
                   else pretrain_trunk(X_pretrain, y_pretrain, hidden_dims=hidden_dims,
                                       dropout=dropout, lr=CONFIG.nn_lr,
                                       weight_decay=weight_decay,
                                       epochs=_quick_pretrain_epochs,
                                       batch_size=CONFIG.nn_pretrain_batch_size,
                                       seed=CONFIG.random_state,
                                       desc=f"Optuna trial {trial.number}: pretrain",
                                       leave=False).trunk.state_dict())

    model = finetune_classifier(
        trunk_state, X_opt_tr, y_opt_tr, hidden_dims=hidden_dims, dropout=dropout,
        lr=lr, weight_decay=weight_decay, freeze_epochs=min(freeze_epochs, 30),
        total_epochs=max(30, min(freeze_epochs + 20, CONFIG.nn_finetune_total_epochs)),
        batch_size=batch_size, seed=CONFIG.random_state,
        desc=f"Optuna trial {trial.number}: fine-tune", leave=False,
    )
    auc = _auc_any(y_opt_val, predict_proba(model, X_opt_val))
    return float(auc) if np.isfinite(auc) else 0.5


if CONFIG.run_optuna:
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=CONFIG.random_state))
    study.optimize(optuna_objective, n_trials=CONFIG.optuna_n_trials, show_progress_bar=True)
    nn_best_params = study.best_params
    print(f"Optuna best validation ROC-AUC = {study.best_value:.3f}")
    print(f"Optuna best params: {nn_best_params}")
    if nn_best_params.get("dropout", 1.0) < 0.1:
        print("\n!! dropout tuned below 0.1 — MC-dropout will report almost no spread.")
        print("   This is the F5 failure repeating: tuning for accuracy kills the")
        print("   dropout-based uncertainty channel. Trust the probability and the")
        print("   reliability diagram (Part E.3), not the MC std.")
else:
    nn_best_params = {"h1": CONFIG.nn_hidden_dims[0], "h2": CONFIG.nn_hidden_dims[1],
                      "dropout": CONFIG.nn_dropout, "lr": CONFIG.nn_finetune_lr,
                      "weight_decay": CONFIG.nn_weight_decay,
                      "freeze_epochs": CONFIG.nn_freeze_epochs,
                      "batch_size": CONFIG.nn_batch_size}
    print("Optuna disabled — using CONFIG defaults:", nn_best_params)

nn_best_params

## 12. Part E.1 — Train the final models

Pretrain the trunk on the full ChEMBL set with the tuned architecture, then fit each
family twice: once on the stratified random training split (for the reference numbers at
the bottom of this section, comparable with F5's) and once on the full PSMA set (the
models that actually score the library), plus stratified bootstrap ensembles.

The reference scores here are *not* the headline. A stratified random split of a set made
of a few congeneric families tests mostly on close analogues of the training compounds,
which is the flaw that F4 quantified and Part E.2 replaces.

In [ ]:
# --- Train the final models ------------------------------------------------------------
nn_hidden_dims_final = (nn_best_params["h1"], nn_best_params["h2"])

pretrained_final = pretrain_trunk(
    X_pretrain, y_pretrain, hidden_dims=nn_hidden_dims_final,
    dropout=nn_best_params["dropout"], lr=CONFIG.nn_lr,
    weight_decay=nn_best_params["weight_decay"], epochs=CONFIG.nn_pretrain_epochs,
    batch_size=CONFIG.nn_pretrain_batch_size, seed=CONFIG.random_state,
    desc="Final trunk pretraining (full ChEMBL set)",
)
_PRETRAINED_STATE = pretrained_final.trunk.state_dict()


def _finetune(X, y, w, trunk_state, seed, desc, leave=False):
    return finetune_classifier(
        trunk_state, X, y, hidden_dims=nn_hidden_dims_final,
        dropout=nn_best_params["dropout"], lr=nn_best_params["lr"],
        weight_decay=nn_best_params["weight_decay"],
        freeze_epochs=nn_best_params["freeze_epochs"],
        total_epochs=CONFIG.nn_finetune_total_epochs,
        batch_size=nn_best_params["batch_size"], sample_weight=w,
        seed=seed, desc=desc, leave=leave,
    )


# reference model on the stratified random split, for comparison with F5
nn_model_eval = _finetune(X_tr, y_tr, w_tr, _PRETRAINED_STATE, CONFIG.random_state,
                          "Fine-tuning reference model (random split)", leave=True)
# deployed model on everything
nn_model_deploy = _finetune(X_train_full, y_train_full, w_train_full, _PRETRAINED_STATE,
                            CONFIG.random_state,
                            "Fine-tuning deploy model (full PSMA set)", leave=True)

gbm_model_eval = fit_lgbm_classifier(X_tr, y_tr, gbm_best_params, sample_weight=w_tr,
                                     seed=CONFIG.random_state)
gbm_model_deploy = fit_lgbm_classifier(X_train_full, y_train_full, gbm_best_params,
                                       sample_weight=w_train_full, seed=CONFIG.random_state)
gbm_bootstrap = bootstrap_lgbm_classifiers(X_train_full, y_train_full, gbm_best_params,
                                           n_bootstrap=CONFIG.n_bootstrap,
                                           sample_weight=w_train_full,
                                           seed=CONFIG.random_state)

kpls_model_eval = fit_kpls_classifier(X_tr, y_tr, kpls_best_params, sample_weight=w_tr,
                                      seed=CONFIG.random_state)
kpls_model_deploy = fit_kpls_classifier(X_train_full, y_train_full, kpls_best_params,
                                        sample_weight=w_train_full, seed=CONFIG.random_state)
kpls_bootstrap = bootstrap_kpls_classifiers(X_train_full, y_train_full, kpls_best_params,
                                            n_bootstrap=CONFIG.n_bootstrap,
                                            sample_weight=w_train_full,
                                            seed=CONFIG.random_state)

print(f"Final models trained. Bootstrap ensembles: "
      f"{len(gbm_bootstrap)} LightGBM, {len(kpls_bootstrap)} KPLS-DA.")

# --- reference-split scores, directly comparable with F5's random-split numbers --------
_ref_probas = {
    "NN (transfer)": predict_proba(nn_model_eval, X_te),
    "LightGBM": lgbm_predict_proba(gbm_model_eval, X_te),
    "KPLS-DA": kpls_predict_proba(kpls_model_eval, X_te),
}
_ref_probas["Blend (equal w)"] = blend_proba_equal(list(_ref_probas.values()))

ref_rows = [{"model": k, **classification_metrics(y_te, v)} for k, v in _ref_probas.items()]
ref_eval_df = pd.DataFrame(ref_rows).set_index("model").round(3)
print("\nSingle stratified random split — reference only. Part E.2's leave-one-cluster-out")
print("is the evaluation that reflects prospective use:\n")
display(ref_eval_df)

## 13. Part E.2 — Leave-one-cluster-out evaluation

The change that matters most in F6, and the reason the F4/F5 numbers were misleading
rather than merely optimistic.

F4 evaluated with repeated `GroupShuffleSplit` draws over Butina clusters and reported
mean ± s.d. The pIC50 result was R² = 0.173 ± 0.270. That spread was not sampling noise —
it was **two different regimes being averaged together**. Butina cluster 0 holds 94
compounds, the largest family in the set, and 89 of them come from US20250213705, which
supplies 119 of the 681 compounds overall. A 20% test draw placed that cluster in the
held-out half in roughly one or two repeats out of ten, and in those repeats the model
lost a quarter of its labels *and* faced a family with no analogue in training. Working
backwards from the numbers: eight repeats near 0.30 plus two near −0.35 average to 0.17
with a wide spread. For KPLS, two repeats near −1.1 produce the reported −0.026 ± 0.579.
The mean described neither regime, and the s.d. was reporting a bimodality as though it
were variance.

So instead of random draws over clusters, **every sizeable cluster gets its own fold,
reported individually**. Clusters of at least `loco_min_cluster_size` compounds — up to
`loco_max_clusters` of them — are held out one at a time; everything smaller is pooled
into a single "remainder" fold of singletons and small groups. Each fold reports its size,
its active/inactive balance, its dominant source tag, and its mean Tanimoto distance to
its own training set. That turns one uninformative average into a statement of the form
"generalises to families A, B and C, fails on D" — which is both more useful and more
honest.

Two aggregate numbers are reported underneath. **Pooled** concatenates every fold's
out-of-fold probability into one vector and scores it once; this is the proper
cross-validated estimate and the one to quote. **Macro** averages the per-fold scores,
which weights a 12-compound fold the same as a 94-compound one and is shown only to make
the spread across families visible. A stratified random-split evaluation runs alongside,
purely so the gap to F5's methodology stays measurable.

The paired pretraining ablation from F4 comes too: on every fold, two otherwise-identical
networks are fine-tuned, one from the ChEMBL-pretrained trunk and one from random init.
F4 found a small, perfectly consistent benefit on random splits (ΔR² = +0.062, 10/10
repeats, p = 0.002) that vanished entirely on both grouped schemes — pretraining helped
the model interpolate within known families and did nothing for reaching new ones. Part
E.2c checks whether the same pattern holds for classification.

In [ ]:
# --- Build the leave-one-cluster-out folds ---------------------------------------------
from rdkit.ML.Cluster import Butina
from sklearn.model_selection import StratifiedShuffleSplit


def butina_cluster_groups(fps, cutoff: float = 0.4) -> np.ndarray:
    """Butina clustering on Tanimoto distance; integer cluster id per compound."""
    n = len(fps)
    dists: List[float] = []
    for i in tqdm(range(1, n), desc=f"Butina distances (cutoff={cutoff})", leave=False):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend(1.0 - s for s in sims)
    clusters = Butina.ClusterData(dists, n, cutoff, isDistData=True)
    groups = np.empty(n, dtype=int)
    for ci, members in enumerate(clusters):
        for idx in members:
            groups[idx] = ci
    return groups


butina_groups = butina_cluster_groups(train_fps, cutoff=CONFIG.butina_cutoff)
_sizes = pd.Series(butina_groups).value_counts()

print(f"Butina clustering of the {len(train_df):,} modelled compounds "
      f"(cutoff {CONFIG.butina_cutoff}):")
print(f"  {len(_sizes)} clusters | largest {_sizes.iloc[0]} "
      f"({100 * _sizes.iloc[0] / len(butina_groups):.1f}%) | "
      f"median size {int(_sizes.median())} | singletons {int((_sizes == 1).sum())}")

# --- fold construction ------------------------------------------------------------------
_big = [int(c) for c in _sizes.index if _sizes[c] >= CONFIG.loco_min_cluster_size]
_big = _big[:CONFIG.loco_max_clusters]
_in_big = np.isin(butina_groups, _big)

LOCO_FOLDS: List[dict] = []
for c in _big:
    mask = butina_groups == c
    LOCO_FOLDS.append({"fold": f"cluster {c}", "mask": mask})
if (~_in_big).sum() >= CONFIG.loco_min_cluster_size:
    LOCO_FOLDS.append({"fold": "remainder (small clusters & singletons)", "mask": ~_in_big})

# fold metadata: size, class balance, provenance, and distance to its own training set
for f in tqdm(LOCO_FOLDS, desc="Fold metadata", unit="fold"):
    m = f["mask"]
    sub = train_df[m]
    f["n"] = int(m.sum())
    f["n_active"] = int((train_df["y"].values[m] == POSITIVE_CLASS).sum())
    f["base_rate"] = round(f["n_active"] / max(f["n"], 1), 3)
    srcs = sub["source_tag"].value_counts()
    f["dominant_source"] = srcs.index[0] if len(srcs) else "?"
    f["source_share"] = round(float(srcs.iloc[0] / len(sub)), 2) if len(srcs) else np.nan
    fold_fps = [train_fps[i] for i in np.flatnonzero(m)]
    rest_fps = [train_fps[i] for i in np.flatnonzero(~m)]
    d = bulk_min_tanimoto_dissimilarity(fold_fps, rest_fps,
                                        desc=f"  {f['fold']}: distance to train")
    f["mean_tanimoto_to_train"] = round(float(np.mean(d)), 3)

loco_folds_df = pd.DataFrame([{k: v for k, v in f.items() if k != "mask"}
                              for f in LOCO_FOLDS]).set_index("fold")
print(f"\n{len(LOCO_FOLDS)} folds covering "
      f"{sum(f['n'] for f in LOCO_FOLDS):,} of {len(train_df):,} compounds:")
display(loco_folds_df)

_pat = loco_folds_df[(loco_folds_df["source_share"] >= 0.6)
                     & (~loco_folds_df["dominant_source"].isin(GENERIC_SOURCE_TAGS))]
if len(_pat):
    print("Folds dominated by a single filing — these are the ones to read carefully,")
    print("because holding them out removes a large, chemically narrow block of training")
    print("data at the same time as it poses an unfamiliar test set:")
    for name, row in _pat.iterrows():
        print(f"  {name}: {row['n']} compounds, {100 * row['source_share']:.0f}% from "
              f"{row['dominant_source']}, mean Tanimoto distance to train "
              f"{row['mean_tanimoto_to_train']}")

In [ ]:
# --- Run every fold, with the paired pretraining ablation -------------------------------
def _fit_committee(X_tr_f, y_tr_f, w_tr_f, X_te_f, seed, label):
    """Fits all four committee members on one fold's training half and returns their
    probability matrices on its test half."""
    out = {}
    for variant, trunk in [("NN (transfer)", _PRETRAINED_STATE), ("NN (scratch)", None)]:
        m = _finetune(X_tr_f, y_tr_f, w_tr_f, trunk, seed,
                      f"  {label}: {variant}", leave=False)
        out[variant] = predict_proba(m, X_te_f)
    g = fit_lgbm_classifier(X_tr_f, y_tr_f, gbm_best_params, sample_weight=w_tr_f, seed=seed)
    out["LightGBM"] = lgbm_predict_proba(g, X_te_f)
    try:
        k = fit_kpls_classifier(X_tr_f, y_tr_f, kpls_best_params, sample_weight=w_tr_f,
                                seed=seed)
        out["KPLS-DA"] = kpls_predict_proba(k, X_te_f)
    except Exception as e:
        tqdm.write(f"  {label}: KPLS-DA failed ({type(e).__name__}); excluded from blend")
        out["KPLS-DA"] = None
    members = [out["NN (transfer)"], out["LightGBM"]]
    if out["KPLS-DA"] is not None:
        members.append(out["KPLS-DA"])
    out["Blend (equal w)"] = blend_proba_equal(members)
    return out


MODEL_ORDER = ["NN (transfer)", "NN (scratch)", "LightGBM", "KPLS-DA", "Blend (equal w)"]

loco_rows = []
oof = {m: np.full((len(train_df), CONFIG.n_activity_classes), np.nan) for m in MODEL_ORDER}

for f in tqdm(LOCO_FOLDS, desc="Leave-one-cluster-out", unit="fold"):
    m = f["mask"]
    tr_i, te_i = np.flatnonzero(~m), np.flatnonzero(m)
    if len(np.unique(y_train_full[tr_i])) < 2:
        tqdm.write(f"  {f['fold']}: training half is single-class; skipped")
        continue
    seed_f = CONFIG.random_state + 7000 + len(loco_rows)
    probas = _fit_committee(X_train_full[tr_i], y_train_full[tr_i], w_train_full[tr_i],
                            X_train_full[te_i], seed_f, f["fold"])
    for name in MODEL_ORDER:
        p = probas.get(name)
        if p is None:
            continue
        oof[name][te_i] = p
        loco_rows.append({
            "fold": f["fold"], "model": name,
            "dominant_source": f["dominant_source"],
            "mean_tanimoto_to_train": f["mean_tanimoto_to_train"],
            **classification_metrics(y_train_full[te_i], p),
        })

loco_eval_df = pd.DataFrame(loco_rows)
print(f"\nCollected {len(loco_eval_df)} (fold, model) results.")

# --- stratified random-split reference, for comparability with F5 -----------------------
rand_rows = []
_sss = StratifiedShuffleSplit(n_splits=CONFIG.loco_random_reference_repeats,
                              test_size=CONFIG.test_size, random_state=CONFIG.random_state)
for r, (tr_i, te_i) in enumerate(tqdm(list(_sss.split(X_train_full, y_train_full)),
                                       desc="Random-split reference", unit="split")):
    seed_r = CONFIG.random_state + 9000 + r
    probas = _fit_committee(X_train_full[tr_i], y_train_full[tr_i], w_train_full[tr_i],
                            X_train_full[te_i], seed_r, f"random split {r}")
    for name in MODEL_ORDER:
        p = probas.get(name)
        if p is None:
            continue
        rand_rows.append({"repeat": r, "model": name,
                          **classification_metrics(y_train_full[te_i], p)})

random_eval_df = pd.DataFrame(rand_rows)
print(f"Collected {len(random_eval_df)} (repeat, model) random-split results.")

## 13b. Part E.2b — Results, per family and pooled

Three views of the same runs.

**Per fold** is the point of the exercise: one row per held-out family, so a model that
works on four families and fails on a fifth reads as exactly that. `mean_tanimoto_to_train`
says how far each fold sat from its own training set, which is the variable that should
predict difficulty — and the one that connects these numbers to the screening library,
whose compounds sit at 0.73–0.83.

**Pooled** concatenates every fold's out-of-fold probability into one vector and scores
it once. This is the proper cross-validated estimate and the number to quote. It is not
the same as averaging the per-fold scores: pooling weights each compound equally, which
is what you want.

**Macro** averages the per-fold scores, weighting a 12-compound fold like a 94-compound
one. It is reported only because the spread across folds is itself the finding.

Read the pooled ROC-AUC against 0.5 (a coin flip) and the pooled EF@1% against 1.0 (a
worthless ranking). For a screen, the enrichment figures matter more than AUC: they say
what fraction of a shortlist would be real.

In [ ]:
# --- Per-fold, pooled, and macro summaries ---------------------------------------------
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

_metric_cols = ["roc_auc", "pr_auc", "balanced_acc", "mcc", "precision", "recall", "brier"]
_metric_cols = [c for c in _metric_cols if c in loco_eval_df.columns]
_ef_cols = [c for c in loco_eval_df.columns if c.startswith("EF@")]

print("=" * 100)
print("PER-FOLD RESULTS")
print("=" * 100)
_per_fold = (loco_eval_df
             .set_index(["fold", "model"])[["n", "n_active", "base_rate",
                                            "mean_tanimoto_to_train"] + _metric_cols + _ef_cols]
             .round(3))
display(_per_fold)

print("\n" + "=" * 100)
print("POOLED (out-of-fold predictions concatenated and scored once) — QUOTE THESE")
print("=" * 100)
pooled_rows = []
for name in MODEL_ORDER:
    P = oof[name]
    ok = np.isfinite(P).all(axis=1)
    if ok.sum() < 10:
        continue
    met = classification_metrics(y_train_full[ok], P[ok])
    met.update({"model": name, "coverage": int(ok.sum())})
    met.update({f"precision@{n}": precision_at_n(y_train_full[ok], P[ok][:, POSITIVE_CLASS], n)
                for n in CONFIG.precision_at_n})
    pooled_rows.append(met)
pooled_df = pd.DataFrame(pooled_rows).set_index("model").round(3)
display(pooled_df)

print("\n" + "=" * 100)
print("MACRO (mean +/- s.d. of the per-fold scores — shows spread across families)")
print("=" * 100)
macro_df = (loco_eval_df.groupby("model")[_metric_cols + _ef_cols]
            .agg(["mean", "std"]).round(3))
macro_df = macro_df.reindex([m for m in MODEL_ORDER if m in macro_df.index])
display(macro_df)

print("\n" + "=" * 100)
print("RANDOM-SPLIT REFERENCE (F5-style; the gap to pooled is the leakage)")
print("=" * 100)
rand_summary = (random_eval_df.groupby("model")[_metric_cols + _ef_cols]
                .agg(["mean", "std"]).round(3))
rand_summary = rand_summary.reindex([m for m in MODEL_ORDER if m in rand_summary.index])
display(rand_summary)

print("\nHeadline comparison — ROC-AUC:")
for name in MODEL_ORDER:
    if name not in pooled_df.index or name not in rand_summary.index:
        continue
    pl = pooled_df.loc[name, "roc_auc"]
    rd = rand_summary.loc[(name), ("roc_auc", "mean")]
    print(f"  {name:<16s} random {rd:.3f}  ->  leave-one-cluster-out (pooled) {pl:.3f}"
          f"   [drop {pl - rd:+.3f}]")
print("\nThe drop is the share of the random-split score that came from testing on close")
print("analogues of the training compounds. The pooled column is the one that bears on a")
print("screening library of unfamiliar chemistry.")

# --- does fold distance predict fold difficulty? ---------------------------------------
_blend = loco_eval_df[loco_eval_df["model"] == "Blend (equal w)"].dropna(subset=["roc_auc"])
if len(_blend) >= 4:
    r = np.corrcoef(_blend["mean_tanimoto_to_train"], _blend["roc_auc"])[0, 1]
    print(f"\nCorrelation between a fold's mean Tanimoto distance to its training set and")
    print(f"its ROC-AUC: r = {r:+.3f} (n = {len(_blend)} folds).")
    print("A strong negative r means distance-from-training predicts failure, which would")
    print("justify using the Tanimoto veto in Part B.2 as the gate on the screening list.")

## 13c. Part E.2c — Does the ChEMBL pretraining help?

The paired test, repeated for classification. On every fold two otherwise-identical
networks were fine-tuned — same architecture, dropout, learning rate, weight decay,
freeze schedule, batch size, epoch budget and seed — differing only in whether the trunk
started from the ChEMBL-pretrained weights or random init.

F4's regression answer was specific: a small but perfectly consistent benefit on random
splits (ΔR² = +0.062, 10 of 10 repeats, p = 0.002), and nothing detectable on either
grouped scheme (4/10 and 5/10 repeats — coin flips). Pretraining helped the model refine
its judgement about chemistry it already knew and did not help it reach anything new.

Since `nn_best_params` was itself tuned with pretraining in the loop, any bias runs
*towards* the transfer arm, so a null result here is a strong null.

In [ ]:
# --- Paired transfer-vs-scratch ablation -----------------------------------------------
from scipy import stats


def _paired_ablation(df, group_key, metric="roc_auc"):
    piv = df.pivot_table(index=group_key, columns="model", values=metric)
    if "NN (transfer)" not in piv.columns or "NN (scratch)" not in piv.columns:
        return None, None
    d = (piv["NN (transfer)"] - piv["NN (scratch)"]).dropna()
    if len(d) < 3:
        return d, None
    try:
        p = float(stats.wilcoxon(d, zero_method="zsplit").pvalue)
    except Exception:
        p = float("nan")
    return d, p


print("Paired ablation on ROC-AUC (positive = pretraining helped)\n")
ablation_rows = []
for scheme, df, key in [("leave-one-cluster-out", loco_eval_df, "fold"),
                        ("random split", random_eval_df, "repeat")]:
    d, p = _paired_ablation(df, key)
    if d is None or len(d) == 0:
        print(f"  {scheme}: not enough paired results.")
        continue
    ablation_rows.append({
        "scheme": scheme, "n_pairs": int(len(d)),
        "mean_dAUC": round(float(d.mean()), 4),
        "std_dAUC": round(float(d.std()), 4),
        "median_dAUC": round(float(d.median()), 4),
        "favouring_transfer": int((d > 0).sum()),
        "wilcoxon_p": round(p, 4) if p is not None and np.isfinite(p) else np.nan,
    })

ablation_df = pd.DataFrame(ablation_rows).set_index("scheme")
display(ablation_df)

for scheme, row in ablation_df.iterrows():
    verdict = ("supports transfer learning"
               if row["mean_dAUC"] > abs(row["std_dAUC"])
               else "no detectable benefit from pretraining")
    print(f"  {scheme:<24s}: dAUC = {row['mean_dAUC']:+.3f} +/- {row['std_dAUC']:.3f}  "
          f"({int(row['favouring_transfer'])}/{int(row['n_pairs'])}, "
          f"p={row['wilcoxon_p']})  ->  {verdict}")

# pooled view: one AUC per arm over all out-of-fold predictions
print("\nPooled out-of-fold ROC-AUC by arm:")
for name in ("NN (transfer)", "NN (scratch)"):
    P = oof[name]
    ok = np.isfinite(P).all(axis=1)
    print(f"  {name:<16s}: {_auc_any(y_train_full[ok], P[ok]):.3f}  "
          f"(n={int(ok.sum())})")
print("\nWith a handful of folds the Wilcoxon test has little power, so weigh the effect")
print("size and its spread first and treat the p-value as a sanity check. A mean whose")
print("standard deviation dwarfs it is a null result whatever the p-value says.")

## 13d. Part E.2d — Is the probability honest?

The reason for switching away from regression. F5 reported ±0.3 log-unit error bars on
compounds whose measured error in that regime was ~1.2 — overconfident by roughly four
times, and undetectably so, because its calibration plot only ever saw in-domain
compounds.

A probability can be checked directly against observed frequency: among compounds the
model calls 70% likely to be active, about 70% should be active. The reliability diagram
below does that on the **pooled out-of-fold predictions**, so every point is a prediction
made for a compound whose whole chemical family was absent from training — the same
situation the screening library presents.

A curve above the diagonal means the model is under-confident; below means
over-confident. The Brier score and expected calibration error (ECE) summarise the gap.
If the pooled curve sits well below the diagonal, the predicted probabilities need
recalibrating (Platt scaling or isotonic regression on held-out folds) before any
shortlist built from them is believable.

In [ ]:
# --- Reliability diagram on pooled out-of-fold predictions -----------------------------
def expected_calibration_error(y_bin, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(p)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi) if hi < 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / n) * abs(y_bin[m].mean() - p[m].mean())
    return float(ece)


_n_models = len([m for m in MODEL_ORDER if np.isfinite(oof[m]).all(axis=1).sum() > 20])
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="perfect calibration")
_cal_rows = []
_colors = plt.get_cmap("tab10")
for i, name in enumerate(MODEL_ORDER):
    P = oof[name]
    ok = np.isfinite(P).all(axis=1)
    if ok.sum() < 20:
        continue
    p = P[ok][:, POSITIVE_CLASS]
    y_bin = (y_train_full[ok] == POSITIVE_CLASS).astype(int)
    n_bins = min(CONFIG.n_calibration_bins * 2, 10)
    try:
        bins = pd.qcut(p, q=n_bins, duplicates="drop")
    except Exception:
        continue
    xs, ys, ns = [], [], []
    for b in bins.categories:
        m = bins == b
        if m.sum() < 5:
            continue
        xs.append(p[m].mean())
        ys.append(y_bin[m].mean())
        ns.append(int(m.sum()))
    if xs:
        ax.plot(xs, ys, "o-", color=_colors(i), linewidth=1.6, markersize=5, label=name)
    _cal_rows.append({
        "model": name, "n": int(ok.sum()),
        "mean_predicted": round(float(p.mean()), 3),
        "observed_rate": round(float(y_bin.mean()), 3),
        "brier": round(float(brier_score_loss(y_bin, np.clip(p, 0, 1))), 4),
        "ECE": round(expected_calibration_error(y_bin, p), 4),
    })
ax.set_xlabel("mean predicted P(active) in bin")
ax.set_ylabel("observed fraction active")
ax.set_title("Reliability — pooled out-of-fold predictions\n(below the diagonal = overconfident)")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# right panel: predicted-probability histograms by true class, for the blend
ax = axes[1]
P = oof["Blend (equal w)"]
ok = np.isfinite(P).all(axis=1)
p = P[ok][:, POSITIVE_CLASS]
y_ok = y_train_full[ok]
for c, name in enumerate(CLASS_NAMES):
    sel = y_ok == c
    if sel.sum():
        ax.hist(p[sel], bins=25, alpha=0.6, label=f"{name} (n={int(sel.sum())})")
ax.set_xlabel("P(active) — blend, out-of-fold")
ax.set_ylabel("compounds")
ax.set_title("Separation between the true classes")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

calibration_df = pd.DataFrame(_cal_rows).set_index("model")
display(calibration_df)
print("`mean_predicted` far from `observed_rate` is a systematic bias; a large ECE with")
print("those two matching means the errors cancel across bins rather than being absent.")
print("Either way, recalibrate on held-out folds before trusting a shortlist's")
print("probabilities as probabilities.")

## 14. Part E.3 — Screen the library

Each committee member returns P(active) for every library compound; the blend is their
equal-weight average. Two spreads come along: the MC-dropout spread of the network, and
the bootstrap-ensemble spread of LightGBM and KPLS-DA. Both describe **model
disagreement**, not predictive error — the honest uncertainty here is the probability
itself, whose quality is measured by Part E.2d's reliability diagram on out-of-fold
predictions.

Every row also carries its applicability-domain flags, now with Tanimoto as a veto.

In [ ]:
# --- Score the screening library --------------------------------------------------------
nn_screen_mean, nn_screen_std = predict_proba_mc_dropout(
    nn_model_deploy, X_screen, CONFIG.nn_mc_dropout_passes)

_gbm_stack = np.stack([lgbm_predict_proba(m, X_screen)
                       for m in tqdm(gbm_bootstrap, desc="GBM bootstrap predictions",
                                      unit="model")], axis=0)
gbm_screen_mean, gbm_screen_std = _gbm_stack.mean(axis=0), _gbm_stack.std(axis=0)

if len(kpls_bootstrap):
    _kpls_stack = np.stack([kpls_predict_proba(m, X_screen)
                            for m in tqdm(kpls_bootstrap, desc="KPLS-DA bootstrap predictions",
                                           unit="model")], axis=0)
    kpls_screen_mean, kpls_screen_std = _kpls_stack.mean(axis=0), _kpls_stack.std(axis=0)
else:
    kpls_screen_mean = kpls_screen_std = None

_members = [nn_screen_mean, gbm_screen_mean]
if kpls_screen_mean is not None:
    _members.append(kpls_screen_mean)
blend_screen = blend_proba_equal(_members)

screen_results = screen_df.copy().reset_index(drop=True)
POS = POSITIVE_CLASS
screen_results["p_active_nn"] = nn_screen_mean[:, POS]
screen_results["p_active_gbm"] = gbm_screen_mean[:, POS]
if kpls_screen_mean is not None:
    screen_results["p_active_kpls"] = kpls_screen_mean[:, POS]
screen_results["p_active"] = blend_screen[:, POS]

# model-disagreement spread, combined in quadrature across the committee
_spreads = [nn_screen_std[:, POS] ** 2, gbm_screen_std[:, POS] ** 2]
if kpls_screen_std is not None:
    _spreads.append(kpls_screen_std[:, POS] ** 2)
screen_results["p_active_spread"] = np.sqrt(np.sum(_spreads, axis=0) / len(_spreads))
screen_results["predicted_class"] = [CLASS_NAMES[i] for i in blend_screen.argmax(axis=1)]

screen_results = pd.concat([screen_results, ad_scores.reset_index(drop=True)], axis=1)

print(f"Scored {len(screen_results):,} library compounds.")
print(f"  P(active) — median {screen_results['p_active'].median():.3f}, "
      f"90th pct {screen_results['p_active'].quantile(0.9):.3f}, "
      f"max {screen_results['p_active'].max():.3f}")
for thr in (0.5, 0.7, 0.9):
    n = int((screen_results["p_active"] >= thr).sum())
    n_in = int(((screen_results["p_active"] >= thr)
                & ~screen_results["consensus_OOD"]).sum())
    print(f"  P(active) >= {thr:.1f}: {n:>7,} compounds "
          f"({n_in:,} of them in-domain)")
screen_results.head()

## 15. Part E.4 — Confidence against domain distance

F5's version of this plot asked whether predicted σ grew with distance from the training
set. It did not, which was the tell that the uncertainty was broken.

The classification equivalent asks something more useful: where do the *confident*
predictions sit relative to the applicability domain? A compound with high P(active) and
low Tanimoto distance is a candidate. A compound with high P(active) and high Tanimoto
distance is the model extrapolating with conviction — and Part E.2's per-fold results say
exactly how much that conviction is worth on unfamiliar chemistry.

In [ ]:
# --- P(active) against domain distance --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

ax = axes[0]
_s = screen_results.sample(min(40000, len(screen_results)), random_state=CONFIG.random_state)
sc = ax.scatter(_s["tanimoto_dissimilarity"], _s["p_active"],
                c=_s["n_flags_OOD"], cmap="viridis", alpha=0.35, s=8, linewidths=0)
ax.axvline(ad.thresholds["tanimoto"], color="#d1495b", linestyle="--", linewidth=1.8,
           label=f"Tanimoto threshold ({ad.thresholds['tanimoto']:.3f})")
ax.set_xlabel("Tanimoto dissimilarity to nearest training compound")
ax.set_ylabel("P(active) — blend")
ax.set_title("Confidence vs domain distance\n(right of the red line = structurally unfamiliar)")
plt.colorbar(sc, ax=ax, label="# AD flags")
ax.legend(fontsize=8, loc="upper left")
ax.grid(alpha=0.2)

ax = axes[1]
_bins = pd.qcut(screen_results["tanimoto_dissimilarity"], q=10, duplicates="drop")
_g = screen_results.groupby(_bins, observed=True)["p_active"]
_mid = [iv.mid for iv in _g.mean().index]
ax.plot(_mid, _g.mean().values, "o-", linewidth=1.8, label="mean P(active)")
ax.fill_between(_mid, _g.quantile(0.1).values, _g.quantile(0.9).values, alpha=0.2,
                label="10th-90th pct")
ax.axvline(ad.thresholds["tanimoto"], color="#d1495b", linestyle="--", linewidth=1.8)
ax.set_xlabel("Tanimoto dissimilarity to nearest training compound (decile midpoint)")
ax.set_ylabel("P(active)")
ax.set_title("Does the model get less confident further from home?")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

_far = screen_results[screen_results["tanimoto_OOD"]]
_near = screen_results[~screen_results["tanimoto_OOD"]]
print(f"Structurally familiar (tanimoto_OOD = False): {len(_near):,} compounds, "
      f"mean P(active) {_near['p_active'].mean():.3f}")
print(f"Structurally unfamiliar (tanimoto_OOD = True): {len(_far):,} compounds, "
      f"mean P(active) {_far['p_active'].mean():.3f}")
if len(_far) and len(_near) and _far["p_active"].mean() >= _near["p_active"].mean():
    print("\n!! The model is no less confident on unfamiliar chemistry than on familiar")
    print("   chemistry. Confidence is therefore not tracking reliability, and the")
    print("   applicability-domain flags - not the probability - are what must gate the")
    print("   shortlist.")

## 16. Part E.5 — Shortlist

Ranked by blended P(active) among in-domain compounds. With `ad_tanimoto_veto` on, "in
domain" now requires passing the structural-similarity test, which every F5 top hit
failed while still being reported as in-domain.

The expected hit rate printed underneath comes from Part E.2's pooled
out-of-fold precision, measured on compounds whose entire chemical family was held out.
That is the closest available estimate of what this shortlist is worth, and it is the
number to quote to anyone deciding whether to put these through an assay.

In [ ]:
# --- Shortlist ---------------------------------------------------------------------------
_cols = ([c for c in ("compound_id", "smiles") if c in screen_results.columns]
         + ["p_active", "p_active_spread", "predicted_class", "n_flags_OOD",
            "consensus_OOD", "tanimoto_dissimilarity"])
_cols += [c for c in ("p_active_nn", "p_active_gbm", "p_active_kpls")
          if c in screen_results.columns]

in_domain = screen_results[~screen_results["consensus_OOD"]]
print(f"{len(in_domain):,} / {len(screen_results):,} compounds are in the applicability "
      f"domain ({100 * len(in_domain) / len(screen_results):.2f}%)")
print(f"  Tanimoto veto: {'ON' if CONFIG.ad_tanimoto_veto else 'OFF'}")
if len(in_domain):
    print(f"  of those, {int((in_domain['p_active'] >= 0.5).sum()):,} have "
          f"P(active) >= 0.5")

top_hits = in_domain.sort_values("p_active", ascending=False)[_cols].head(25)
display(top_hits)

# --- what is this shortlist actually worth? ----------------------------------------------
print("\n" + "=" * 92)
print("EXPECTED YIELD, from the leave-one-cluster-out results")
print("=" * 92)
if "Blend (equal w)" in pooled_df.index:
    _p = pooled_df.loc["Blend (equal w)"]
    print(f"  pooled out-of-fold ROC-AUC        : {_p['roc_auc']:.3f}  (0.5 = coin flip)")
    if "pr_auc" in _p:
        print(f"  pooled out-of-fold PR-AUC         : {_p['pr_auc']:.3f}  "
              f"(base rate = {_p['base_rate']:.3f})")
    for n in CONFIG.precision_at_n:
        k = f"precision@{n}"
        if k in _p and np.isfinite(_p[k]):
            print(f"  precision@{n:<4d}                     : {_p[k]:.3f}  -> of the top "
                  f"{n} ranked, ~{_p[k] * 100:.0f}% were genuinely active when the whole "
                  f"family was unseen")
    for c in [c for c in pooled_df.columns if c.startswith("EF@")]:
        if np.isfinite(_p[c]):
            print(f"  {c:<34s}: {_p[c]:.2f}x enrichment over the base rate")

_md = top_hits["tanimoto_dissimilarity"].median() if len(top_hits) else np.nan
print(f"\n  median Tanimoto distance of this shortlist : {_md:.3f}")
print(f"  AD Tanimoto threshold                      : {ad.thresholds['tanimoto']:.3f}")
if np.isfinite(_md) and _md > ad.thresholds["tanimoto"]:
    print("\n  !! Even after the veto, the shortlist's median distance exceeds the")
    print("     threshold. Treat the ranking as a weak prior, not a prediction.")
print("\n  Carry the precision@N figure, not the probability, into any decision about")
print("  assay time. And note what has NOT been fixed: the screening library still")
print("  contains fragments and solvents, and LigPrep variants of the same compound are")
print("  still separate rows, so expect duplicates in the ranking.")

## 16b. Part E.6 — Which atoms drive P(active)?

Per-atom attribution by **substructure ablation**: each "on" fingerprint bit is set to 0,
the blended P(active) recomputed, and the change spread over the atoms in that bit's
environment.

This is a correction, not just a port. F5 perturbed each on-bit from 1 to 1.05 and divided
by 0.05 — which does nothing to a tree ensemble, because a 0.05 nudge on a binary feature
essentially never crosses a split threshold. LightGBM returned an unchanged prediction and
contributed exactly zero, so the map was an NN gradient wearing a blend's label. The
meaningful counterfactual for a binary fingerprint is 1 → 0 — "what if this substructure
were absent" — and that works identically for trees, kernels and networks.

In [ ]:
# --- Which atoms drive P(active)? --------------------------------------------------------
from rdkit.Chem import Draw
from collections import defaultdict


def get_bit_atoms(mol: Chem.Mol, bit_info: dict, bit_id: int) -> set:
    atoms = set()
    for atom_idx, radius in bit_info[bit_id]:
        if radius == 0:
            atoms.add(atom_idx)
        else:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
            amap: dict = {}
            Chem.PathToSubmol(mol, env, atomMap=amap)
            atoms.update(amap.keys())
    return atoms


def predict_blend_single(x_row: np.ndarray) -> float:
    """Blended P(active) for one feature vector, dropout off."""
    xb = x_row.reshape(1, -1).astype(np.float32)
    members = [predict_proba(nn_model_deploy, xb), lgbm_predict_proba(gbm_model_deploy, xb)]
    try:
        members.append(kpls_predict_proba(kpls_model_deploy, xb))
    except Exception:
        pass
    return float(blend_proba_equal(members)[0, POSITIVE_CLASS])


def compute_atom_sensitivities(mol: Chem.Mol, x_row: np.ndarray) -> Dict[int, float]:
    """Per-atom contribution to P(active), by ABLATING each 'on' fingerprint bit.

    F5 perturbed each on-bit from 1 to 1.05 and divided by 0.05. For a tree ensemble that
    does nothing at all: a 0.05 nudge on a binary feature essentially never crosses a
    split threshold, so LightGBM returned an unchanged prediction and contributed exactly
    zero. The map was an NN gradient wearing a blend's label. The meaningful
    counterfactual for a binary fingerprint is 1 -> 0: "what if this substructure were
    absent". That works identically for trees, kernels and networks.
    """
    _, bit_info = featurizer.morgan_fp_with_bit_info(mol)
    base = predict_blend_single(x_row)
    atom_scores: Dict[int, float] = defaultdict(float)
    on_bits = [b for b in bit_info if x_row[b] > 0.5]
    for b in tqdm(on_bits, desc="  ablating fingerprint bits", unit="bit", leave=False):
        x_pert = x_row.copy()
        x_pert[b] = 0.0
        delta = base - predict_blend_single(x_pert)   # >0 = removing it HURT P(active)
        atoms = get_bit_atoms(mol, bit_info, b)
        if atoms:
            for a in atoms:
                atom_scores[a] += delta / len(atoms)
    return atom_scores


if CONFIG.feature_set == "descriptors_only":
    print("feature_set='descriptors_only' -> no fingerprint bits to attribute; skipped.")
elif len(top_hits) == 0:
    print("No in-domain compounds to attribute.")
else:
    top_n_interp = min(4, len(top_hits))
    fig, axes = plt.subplots(1, top_n_interp, figsize=(4.2 * top_n_interp, 4.4))
    axes = np.atleast_1d(axes)
    cmap = plt.get_cmap("coolwarm")
    for ax, row_idx in zip(axes, tqdm(list(top_hits.index[:top_n_interp]),
                                       desc="Atom-contribution maps", unit="mol")):
        mol = screen_feat["mols"][row_idx]
        atom_scores = compute_atom_sensitivities(mol, X_screen[row_idx])
        if not atom_scores:
            ax.set_title("no 'on' bits found"); ax.axis("off"); continue
        max_abs = max(abs(v) for v in atom_scores.values()) or 1.0
        colors = {a: cmap(0.5 + 0.5 * (v / max_abs))[:3] for a, v in atom_scores.items()}
        img = Draw.MolToImage(mol, size=(320, 320), highlightAtoms=list(atom_scores),
                              highlightAtomColors=colors)
        ax.imshow(img); ax.axis("off")
        cid = (screen_results.loc[row_idx, "compound_id"]
               if "compound_id" in screen_results.columns else row_idx)
        ax.set_title(f"{cid}\nP(active)={screen_results.loc[row_idx, 'p_active']:.3f}",
                     fontsize=9)
    plt.suptitle("Atom contributions to P(active) by substructure ablation "
                 "(red = raises P(active), blue = lowers it)", y=1.03)
    plt.tight_layout()
    plt.show()
    print("Each on-bit is set to 0 and the blended P(active) recomputed; the change is")
    print("spread over the atoms in that bit's environment. Unlike F5's 1 -> 1.05 nudge,")
    print("every committee member responds to this, so the map reflects the blend.")

## 17. Part F — Save outputs

Everything needed to reproduce, audit or redeploy this run: the shortlist and full library
predictions, the classification dataset with its source tags and label provenance, all
three model families with their bootstrap ensembles, the shared feature scaler, the
descriptor selector, the applicability-domain object, and every evaluation table.

The **out-of-fold predictions** are saved too. Every pooled number in Part E.2 derives
from them, so the calibration, enrichment and precision@N figures can be recomputed — or
challenged — without re-running the folds.

In [ ]:
# --- Save everything --------------------------------------------------------------------
import joblib

out_dir = Path(CONFIG.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

screen_results.to_csv(out_dir / "screening_predictions_classification.csv", index=False)
top_hits.to_csv(out_dir / "shortlist.csv", index=False)
train_df.to_csv(out_dir / "psma_classification_dataset.csv", index=False)

# evaluation
loco_eval_df.to_csv(out_dir / "loco_per_fold.csv", index=False)
loco_folds_df.to_csv(out_dir / "loco_fold_definitions.csv")
pooled_df.to_csv(out_dir / "loco_pooled.csv")
macro_df.to_csv(out_dir / "loco_macro.csv")
random_eval_df.to_csv(out_dir / "random_split_reference.csv", index=False)
ablation_df.to_csv(out_dir / "transfer_learning_ablation.csv")
calibration_df.to_csv(out_dir / "probability_calibration.csv")
if len(derived_report):
    derived_report.to_csv(out_dir / "derived_endpoint_report.csv", index=False)

# out-of-fold probabilities — the basis of every pooled number above, kept so the
# calibration and enrichment figures can be re-derived without re-running the folds
_oof = pd.DataFrame({"smiles": train_df["smiles"].values, "y": y_train_full,
                     "source_tag": train_df["source_tag"].values,
                     "butina_cluster": butina_groups})
for name in MODEL_ORDER:
    _oof[f"oof_p_active::{name}"] = oof[name][:, POSITIVE_CLASS]
_oof.to_csv(out_dir / "out_of_fold_predictions.csv", index=False)

# models
torch.save({
    "trunk_state_dict": {k: v.cpu() for k, v in nn_model_deploy.trunk.state_dict().items()},
    "clf_head_state_dict": {k: v.cpu() for k, v in nn_model_deploy.clf_head.state_dict().items()},
    "hidden_dims": nn_hidden_dims_final,
    "dropout": nn_best_params["dropout"],
    "n_classes": CONFIG.n_activity_classes,
    "class_names": CLASS_NAMES,
    "primary_endpoint": PRIMARY,
    "active_threshold": CONFIG.active_threshold,
}, out_dir / "nn_transfer_classifier.pt")

joblib.dump(gbm_model_deploy, out_dir / "lightgbm_classifier.joblib")
joblib.dump(gbm_bootstrap, out_dir / "lightgbm_bootstrap_ensemble.joblib")
joblib.dump(kpls_model_deploy, out_dir / "kpls_da_classifier.joblib")
joblib.dump(kpls_bootstrap, out_dir / "kpls_da_bootstrap_ensemble.joblib")
joblib.dump(feature_scaler, out_dir / "feature_scaler.joblib")
joblib.dump(descriptor_selector, out_dir / "descriptor_selector.joblib")
joblib.dump(ad, out_dir / "applicability_domain.joblib")

with open(out_dir / "config.json", "w") as fh:
    json.dump({k: (list(v) if isinstance(v, (tuple, list)) else v)
               for k, v in CONFIG.__dict__.items()}, fh, indent=2, default=str)
with open(out_dir / "tuning_summary.json", "w") as fh:
    json.dump({
        "gbm_best_params": gbm_best_params,
        "gbm_candidates": {k: {"autoqsar": v[0], "params": v[1],
                               "train_auc": v[2], "test_auc": v[3]}
                           for k, v in gbm_tuning_results.items()},
        "kpls_best_params": kpls_best_params,
        "nn_best_params": nn_best_params,
        "feature_set": CONFIG.feature_set,
    }, fh, indent=2, default=str)

print(f"Saved to {out_dir}/:")
for p in sorted(out_dir.iterdir()):
    print(f" - {p.name}")

## 18. Notes for reuse

### Reading the results

- **Quote the pooled leave-one-cluster-out numbers**, not the random-split reference and
  not the macro average. Pooled scores every compound once on a prediction made while its
  whole chemical family was absent from training, which is the situation a screening
  library presents. The random-split row is kept only so the gap to F5's methodology
  stays visible, and the macro row only so the spread across families does.
- **For a screen, read precision@N and EF before AUC.** "Of the top 200 ranked, this
  fraction were really active" is what decides whether a shortlist earns assay time. AUC
  can look respectable while the top of the ranking is worthless.
- **Check Part E.2d before believing any probability.** If the reliability curve sits
  below the diagonal, recalibrate (Platt or isotonic on held-out folds) before quoting
  P(active) as a probability to anyone.
- **A fold that fails is information, not noise.** The per-fold table names which family
  the model could not reach. Averaging that into a single number is what made the F4
  results misleading.

### Knobs worth turning

- `active_threshold` (default 8.0, i.e. 10 nM) — pre-registered from pharmacology and
  checked against the distribution in Part A.2b. Change it before looking at results, not
  after; and if you do, re-run Part A.2b to confirm the classes stay usable.
- `n_activity_classes = 3` switches to active / moderate / inactive at 8.0 / 6.0. Viable
  for pIC50 (128 / 215 / 141) and not for pKi (30 inactives). Note that three classes
  trained with plain cross-entropy treats confusing active-for-inactive as no worse than
  active-for-moderate; if you go this way, consider an ordinal formulation (two cumulative
  binary models, P(≥moderate) and P(≥active)) instead.
- `ad_tanimoto_veto` (default True) — makes structural dissimilarity decisive rather than
  one vote in four. Turning it off reproduces F5's much larger and much less trustworthy
  in-domain set.
- `downweight_dominant_source` (default False) — inverse-sqrt-frequency weights per
  source tag. One patent supplies ~26% of the labels; whether to discount it is a
  judgement, so it is exposed rather than assumed. Compare the pooled numbers both ways.
- `butina_cutoff` (default 0.4) — raising it towards 0.6 makes families coarser and the
  held-out test harder.

### A new target instead of PSMA

Change `target_uniprot_id` / `target_name_like` / `organism_like`, **and**
`chembl_pretrain_target_keywords` to describe that target's related family. Then re-run
Part A.2b before anything else: the thresholds, the class balance, and whether the two
endpoints are independent are all target-specific, and all three were wrong-by-default
for PSMA.

### Still outstanding

- **The screening library is the wrong file.** Its first entries are butyl chloride,
  β-propiolactone and diethylene glycol — solvents and fragments, not a drug-like
  commercial collection. Everything downstream inherits that.
- **LigPrep variants are not merged.** F5's top-15 contained `sdf:71638` and `sdf:43804`
  twice each, as different protonation or stereo forms of one compound. Deduplicate on
  standardised InChIKey before ranking.
- **Compound deduplication still happens on raw SMILES** before RDKit standardisation, in
  both the training set and the library, so salt and tautomer variants survive as separate
  rows and can straddle a fold boundary.
- **Nothing has been validated in a lab.** Every number here is the model grading its own
  homework under progressively stricter exam conditions. The leave-one-cluster-out figures
  should be read as a prediction of a low hit rate, and the only thing that settles it is
  measuring a handful of compounds.
- **More and more varied data is the real lever.** 452 numeric labels from a few families,
  a quarter of them one patent, is not enough to teach generalisable chemistry — which is
  what Part E.2 measures and what no amount of re-framing the target will fix. PSMA actives
  from ChEMBL that are absent from BindingDB, and property-matched decoys as presumed
  inactives, would both do more than any modelling change in this notebook.